##### About this notebook:

In [ ]:
#-----------------------------------------------------------------------------------------------------------------------------
# Author:             Erick Rico Esparza
# Dates:              Feb 21 - Mar 11, 2026
# Description:        This notebook builds on Stage 1–2 outputs; it does not redo them, it reorganizes them
# Organization:       Tampere University / Institute of Atmospheric Sciences and Climate Change (ICAyCC-UNAM)
#-----------------------------------------------------------------------------------------------------------------------------

# Stage 3

## 1. Libraries and setup

In [97]:
# --- Core
import numpy as np
import pandas as pd
import xarray as xr
from xarray.coding.variables import SerializationWarning

import os
from pathlib import Path
import warnings

# --- Plotting
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind
import matplotlib.patches as mpatches
from matplotlib.colors import TwoSlopeNorm
import calendar
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
import matplotlib.ticker as mticker
from scipy.stats import ttest_1samp

# --- Mapping
import cartopy.crs as ccrs
import cartopy.feature as cfeature

In [35]:
warnings.filterwarnings("ignore", category=SerializationWarning)

In [3]:
# Display options
pd.options.display.float_format = '{:,.2f}'.format
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

## 2. Paths, I/O rules, and folder structure

This notebook lives inside `Schedule/Stage 3/`.
All core inputs (CSV + 500 hPa NetCDF files) are located **one level above** (i.e., in `Schedule/`).

Using `Path` objects and relative paths to keep the project portable.


In [8]:
# --- Project root
HERE = Path.cwd()  # running from within Stage 3
PARENT = HERE.parent  # Schedule/

# 1. Inputs (one level above Stage 3)
PM_CSV = PARENT / "pm_cdmx_citymean_daily_2012_2024.csv"
HGT_NC = PARENT / "hgt500_mex_2012_2024.nc"
U_NC   = PARENT / "uwnd500_mex_2012_2024.nc"
V_NC   = PARENT / "vwnd500_mex_2012_2024.nc"

# --- Precipitation (dummy download) - placeholder filename
PRECIP_NC = PARENT / "precip_sfc_mex_2012_2024.nc"

# --- Stage 2 outputs (tables) to reuse in Stage 3
STAGE2_TAB = PARENT / "Stage 2" / "outputs" / "tables"

PM_P90_FLAGS_CSV = STAGE2_TAB / "pm_daily_with_p90_flags_2012_2024.csv"
P90_THR_CSV      = STAGE2_TAB / "p90_thresholds_by_calendar_month_2012_2024.csv"

# --- Sanity check: make sure we are running from within "Stage 3"
print(f"HERE:   {HERE}")
print(f"PARENT: {PARENT}")

if "Stage 3" not in HERE.name:
    warnings.warn(
        f"You are running from '{HERE.name}'. Expected to run from inside a folder named like 'Stage 3'. "
        "Paths may be wrong."
    )

# 2. Outputs (inside Stage 3)
OUT_FIG = HERE / "outputs" / "figures"
OUT_TAB = HERE / "outputs" / "tables"

OUT_FIG.mkdir(parents=True, exist_ok=True)
OUT_TAB.mkdir(parents=True, exist_ok=True)

HERE:   c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 3
PARENT: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule


In [9]:
# 3. Existence check
paths = {
    "PM_CSV": PM_CSV,
    "HGT_NC": HGT_NC,
    "U_NC": U_NC,
    "V_NC": V_NC,
    "PRECIP_NC": PRECIP_NC,
    "STAGE2_TAB": STAGE2_TAB,
    "PM_P90_FLAGS_CSV": PM_P90_FLAGS_CSV,
    "P90_THR_CSV": P90_THR_CSV,
    "OUT_FIG": OUT_FIG,
    "OUT_TAB": OUT_TAB,
}

for k, p in paths.items():
    print(f"{k}: {p}  | exists={p.exists()}")

PM_CSV: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\pm_cdmx_citymean_daily_2012_2024.csv  | exists=True
HGT_NC: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\hgt500_mex_2012_2024.nc  | exists=True
U_NC: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\uwnd500_mex_2012_2024.nc  | exists=True
V_NC: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\vwnd500_mex_2012_2024.nc  | exists=True
PRECIP_NC: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\precip_sfc_mex_2012_2024.nc  | exists=False
STAGE2_TAB: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 2\outputs\tables  | exists=True
PM_P90_FLAGS_CSV: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 2\outputs\tables\pm_daily_with_p90_flags_2012_2024.csv  | exists=True
P90_THR_CSV: c:\Users\DELL\OneDrive - TUNI

In [6]:
# 4. I/O rules
"""
I/O rules for Stage 3
- Inputs are read from one level above (Schedule/).
- Outputs are saved inside Stage 3/outputs/:
    - figures -> Stage 3/outputs/figures/  (PNG, dpi=250)
    - tables  -> Stage 3/outputs/tables/   (CSV)
- Notebook output stays clean:
    - do not print entire DataFrames
    - after saving, print only a short confirmation like: "saved: <path>"
- Filenames: using consistent prefixes:
    - figs: stage3_<topic>_<pol>_<month-or-season>.png
    - tabs: stage3_<topic>_<pol>.csv
"""
print("I/O rules loaded: outputs -> Stage 3/outputs/{figures,tables}.")

I/O rules loaded: outputs -> Stage 3/outputs/{figures,tables}.


In [7]:
# --- Domain & global constants

# Regional domain for composites (Mexico-centered box)
LON_MIN, LON_MAX = -120.0, -85.0
LAT_MIN, LAT_MAX = 12.0, 33.0

# CDMX reference point (marker on maps)
LON_CDMX, LAT_CDMX = -99.13, 19.43

# Valley of Mexico regional box (renamed from MCMA box)
SW_lat, SW_lon = 18.3, -100.9
NE_lat, NE_lon = 20.7, -97.4

VOM_BOX = (
    SW_lon,
    SW_lat,
    NE_lon - SW_lon,
    NE_lat - SW_lat
)

print("Domain and Valley of Mexico box defined.")

# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

Domain and Valley of Mexico box defined.


## 3. Load daily PM + event tables (from Stage 2 outputs)

### 3.1 Load base daily PM (raw)

In [10]:
# --- Load PM data (raw city-mean daily)
df_pm = pd.read_csv(PM_CSV)

# Parse dates and standardize column names
df_pm["DATE"] = pd.to_datetime(df_pm["DATE"])
df_pm = df_pm.sort_values("DATE").set_index("DATE")

# Ensure numeric (coerce errors to NaN)
for col in ["PM10", "PM2.5"]:
    df_pm[col] = pd.to_numeric(df_pm[col], errors="coerce")

print(df_pm.head())
print(df_pm.tail())
print(df_pm.info())
print("\nDate range:", df_pm.index.min(), "to", df_pm.index.max())
print("\nMissing values:\n", df_pm[["PM10", "PM2.5"]].isna().sum())

# QC summary table
qc = pd.DataFrame({
    "start_date": [df_pm.index.min()],
    "end_date": [df_pm.index.max()],
    "n_days": [df_pm.shape[0]],
    "pm10_missing": [df_pm["PM10"].isna().sum()],
    "pm25_missing": [df_pm["PM2.5"].isna().sum()],
})

qc

             PM10  PM2.5
DATE                    
2012-01-01 100.14  66.43
2012-01-02  19.29   6.14
2012-01-03  38.00  17.43
2012-01-04  67.71  35.00
2012-01-05  61.43  28.86
            PM10  PM2.5
DATE                   
2024-12-27 42.86  22.71
2024-12-28 47.83  23.00
2024-12-29 43.83  21.67
2024-12-30 51.40  26.00
2024-12-31 52.00  26.00
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 4555 entries, 2012-01-01 to 2024-12-31
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   PM10    4555 non-null   float64
 1   PM2.5   4555 non-null   float64
dtypes: float64(2)
memory usage: 106.8 KB
None

Date range: 2012-01-01 00:00:00 to 2024-12-31 00:00:00

Missing values:
 PM10     0
PM2.5    0
dtype: int64


,start_date,end_date,n_days,pm10_missing,pm25_missing
0,2012-01-01,2024-12-31,4555,0,0


### 3.2 Load Stage 2 working table (p90 thresholds + flags)

In [11]:
# --- Load Stage 2 event table (p90 thresholds + flags from Stage 2 outputs)
df_evt = pd.read_csv(PM_P90_FLAGS_CSV)

# Parse dates and standardize
df_evt["DATE"] = pd.to_datetime(df_evt["DATE"])
df_evt = df_evt.sort_values("DATE").set_index("DATE")

# Ensure numeric (coerce errors to NaN)
for col in ["PM10", "PM2.5", "thr_p90_PM10", "thr_p90_PM2.5", "evt_PM10_p90", "evt_PM2.5_p90"]:
    if col in df_evt.columns:
        df_evt[col] = pd.to_numeric(df_evt[col], errors="coerce")

print(df_evt.head())
print(df_evt.tail())
print(df_evt.info())
print("\nDate range:", df_evt.index.min(), "to", df_evt.index.max())

# Missing values (core variables)
core_cols = ["PM10", "PM2.5", "thr_p90_PM10", "thr_p90_PM2.5", "evt_PM10_p90", "evt_PM2.5_p90"]
print("\nMissing values:\n", df_evt[core_cols].isna().sum())

             PM10  PM2.5  thr_p90_PM10  thr_p90_PM2.5  evt_PM10_p90  evt_PM2.5_p90
DATE                                                                              
2012-01-01 100.14  66.43         78.89          41.00             1              1
2012-01-02  19.29   6.14         78.89          41.00             0              0
2012-01-03  38.00  17.43         78.89          41.00             0              0
2012-01-04  67.71  35.00         78.89          41.00             0              0
2012-01-05  61.43  28.86         78.89          41.00             0              0
            PM10  PM2.5  thr_p90_PM10  thr_p90_PM2.5  evt_PM10_p90  evt_PM2.5_p90
DATE                                                                             
2024-12-27 42.86  22.71         84.71          43.58             0              0
2024-12-28 47.83  23.00         84.71          43.58             0              0
2024-12-29 43.83  21.67         84.71          43.58             0              0
2024-12-3

### 3.3 Load Stage 2 p90 threshold table (calendar-month) + quick view

In [12]:
# --- Load Stage 2 p90 thresholds by calendar month (reporting/diagnostics)
thr_p90 = pd.read_csv(P90_THR_CSV)

print(thr_p90.head())
print(thr_p90.tail())
print(thr_p90.info())

# Basic check: expect 12 rows (months)
if "month" in thr_p90.columns:
    print("\nMonths present:", sorted(thr_p90["month"].unique()))
else:
    warnings.warn("Column 'month' not found in thr_p90 table. Please verify the CSV structure.")

   month month_name  p90_PM10  p90_PM2.5
0      1        Jan     78.89      41.00
1      2        Feb     75.97      35.70
2      3        Mar     68.29      32.75
3      4        Apr     69.63      37.13
4      5        May     72.88      40.26
    month month_name  p90_PM10  p90_PM2.5
7       8        Aug     42.33      23.29
8       9        Sep     44.01      26.17
9      10        Oct     52.50      28.00
10     11        Nov     67.05      35.99
11     12        Dec     84.71      43.58
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   month       12 non-null     int64  
 1   month_name  12 non-null     object 
 2   p90_PM10    12 non-null     float64
 3   p90_PM2.5   12 non-null     float64
dtypes: float64(2), int64(1), object(1)
memory usage: 516.0+ bytes
None

Months present: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5),

### 3.4 Add month column to df_evt + sanity checks (p90 structure)

In [13]:
# --- Add month column and sanity checks
df_evt["month"] = df_evt.index.month

required_cols = [
    "PM10", "PM2.5",
    "thr_p90_PM10", "thr_p90_PM2.5",
    "evt_PM10_p90", "evt_PM2.5_p90",
    "month"
]

missing = [c for c in required_cols if c not in df_evt.columns]
if missing:
    raise KeyError(f"Stage 2 event table is missing required columns: {missing}")

# Month range check
assert df_evt["month"].between(1, 12).all(), "Month column has values outside 1..12."

# Flags should be 0/1 (allow NaN if any)
for f in ["evt_PM10_p90", "evt_PM2.5_p90"]:
    vals = sorted(df_evt[f].dropna().unique())
    if not set(vals).issubset({0, 1}):
        warnings.warn(f"Unexpected values in {f}: {vals} (expected only 0/1)")

print("OK: df_evt has required p90 columns + valid month + flags look like 0/1.")

OK: df_evt has required p90 columns + valid month + flags look like 0/1.


### 3.5 Prepare p10 placeholders (no computation yet)

In [14]:
# --- (filled in Section 7)
p10_cols = [
    "thr_p10_PM10",
    "thr_p10_PM2.5",
    "evt_PM10_p10",
    "evt_PM2.5_p10",
]

for c in p10_cols:
    if c not in df_evt.columns:
        df_evt[c] = np.nan

print("Prepared p10 placeholder columns in df_evt (to be filled in Section 7).")
print(df_evt[p10_cols].head())

Prepared p10 placeholder columns in df_evt (to be filled in Section 7).
            thr_p10_PM10  thr_p10_PM2.5  evt_PM10_p10  evt_PM2.5_p10
DATE                                                                
2012-01-01           NaN            NaN           NaN            NaN
2012-01-02           NaN            NaN           NaN            NaN
2012-01-03           NaN            NaN           NaN            NaN
2012-01-04           NaN            NaN           NaN            NaN
2012-01-05           NaN            NaN           NaN            NaN


### 3.6 Alignment check (raw PM vs Stage 2 df)

In [15]:
# --- df_pm vs df_evt
if df_pm.index.min() != df_evt.index.min() or df_pm.index.max() != df_evt.index.max():
    warnings.warn(
        "Date ranges differ between df_pm (raw) and df_evt (Stage 2). "
        "Stage 3 will proceed using df_evt as the canonical daily table."
    )

# Check basic equality of PM columns on overlapping dates (quick sanity)
common_idx = df_pm.index.intersection(df_evt.index)
if len(common_idx) > 0:
    diff_pm10 = (df_pm.loc[common_idx, "PM10"] - df_evt.loc[common_idx, "PM10"]).abs().max()
    diff_pm25 = (df_pm.loc[common_idx, "PM2.5"] - df_evt.loc[common_idx, "PM2.5"]).abs().max()
    print(f"Max abs difference (PM10) on overlap: {diff_pm10}")
    print(f"Max abs difference (PM2.5) on overlap: {diff_pm25}")
else:
    warnings.warn("No overlapping dates between df_pm and df_evt for comparison.")

print("\nLoaded daily PM tables. Canonical table for Stage 3 onward: df_evt")

Max abs difference (PM10) on overlap: 0.0
Max abs difference (PM2.5) on overlap: 0.0

Loaded daily PM tables. Canonical table for Stage 3 onward: df_evt


### 3.7 PM date completeness check

In [24]:
# --- Missing dates in daily PM table (df_evt)
# Assumes df_evt has a DatetimeIndex (DATE) and has already been sliced to the overlap window.

full = pd.date_range(df_evt.index.min(), df_evt.index.max(), freq="D")
missing = full.difference(df_evt.index)

print("PM expected daily range:", full.min().date(), "to", full.max().date())
print("Expected #days:", len(full))
print("Available #days in df_evt:", df_evt.shape[0])
print("Missing #days:", len(missing))

# Show a preview + save full list to CSV for documentation
if len(missing) > 0:
    print("\nFirst 20 missing dates:")
    print(missing[:20])

    miss_df = pd.DataFrame({"missing_DATE": missing})
    out_csv = OUT_TAB / "stage3_missing_pm_dates.csv"
    miss_df.to_csv(out_csv, index=False)
    print(f"\nsaved: {out_csv}")
else:
    print("\nNo missing dates detected (complete daily coverage).")

PM expected daily range: 2012-01-01 to 2024-12-31
Expected #days: 4749
Available #days in df_evt: 4555
Missing #days: 194

First 20 missing dates:
DatetimeIndex(['2022-01-02', '2022-01-03', '2022-01-04', '2022-01-05', '2022-01-06', '2022-01-07', '2022-01-08',
               '2022-01-09', '2022-01-10', '2022-01-11', '2022-01-12', '2022-01-13', '2022-01-14', '2022-01-15',
               '2022-01-16', '2022-01-17', '2022-01-18', '2022-01-19', '2022-01-20', '2022-01-21'],
              dtype='datetime64[ns]', freq=None)

saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 3\outputs\tables\stage3_missing_pm_dates.csv


In [27]:
# Table of missing values ​​per month
if len(missing) > 0:
    miss_by_month = miss_df.copy()
    miss_by_month['year'] = miss_by_month['missing_DATE'].dt.year
    miss_by_month['month'] = miss_by_month['missing_DATE'].dt.month
    
    # Group by year and month
    missing_monthly = miss_by_month.groupby(['year', 'month']).size().reset_index(name='missing_days')
    
    # Add month name
    missing_monthly['month_name'] = missing_monthly['month'].apply(
        lambda x: ['January', 'February', 'March', 'April', 'May', 'June',
                   'July', 'August', 'September', 'October', 'November', 'December'][x-1]
    )
    
    # Reorganize columns
    missing_monthly = missing_monthly[['year', 'month', 'month_name', 'missing_days']]
    
    print("\nMissing values ​​per month:")
    print(missing_monthly.to_string(index=False))
    
    # Save monthly summary to CSV
    out_csv_monthly = OUT_TAB / "stage3_missing_pm_dates_by_month.csv"
    missing_monthly.to_csv(out_csv_monthly, index=False)
    print(f"\nSaved: {out_csv_monthly}")
    
    # Summary by year
    missing_yearly = miss_by_month.groupby('year').size().reset_index(name='missing_days')
    print("\nSummary by year:")
    print(missing_yearly.to_string(index=False))
else:
    print("No missing values to tabulate.")


Missing values ​​per month:
 year  month month_name  missing_days
 2022      1    January            30
 2022      2   February            15
 2022      3      March             2
 2022      4      April             9
 2022      5        May            31
 2022      6       June            30
 2022      7       July             5
 2022      8     August             7
 2022      9  September            20
 2022     10    October            26
 2022     11   November            16
 2022     12   December             3

Saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 3\outputs\tables\stage3_missing_pm_dates_by_month.csv

Summary by year:
 year  missing_days
 2022           194


## 4. Load reanalysis (500 hPa) + preprocessing

### 4.1 Helpers: open + subset curvilinear domain

In [16]:
def open_da(path: Path, varname: str) -> xr.DataArray:
    """
    Open a NetCDF and return the requested variable as a DataArray.
    Assumes the dataset contains the relevant 500 hPa field with coords including time and 2D lat/lon.
    """
    ds = xr.open_dataset(path)
    da = ds[varname].sortby("time")
    return da


def subset_domain_curvilinear(
    da: xr.DataArray,
    lat_min: float, lat_max: float,
    lon_min: float, lon_max: float
) -> xr.DataArray:
    """
    Subset a curvilinear grid where lat/lon are 2D coords with dims (y, x).
    Uses a boolean mask and drop=True.
    """
    # Standardize lon to [-180, 180] if dataset uses 0..360
    if "lon" in da.coords:
        lon = da["lon"]
        if float(lon.max()) > 180.0:
            da = da.assign_coords(lon=(((lon + 180) % 360) - 180))

    lat = da["lat"]
    lon = da["lon"]

    mask = (lat >= lat_min) & (lat <= lat_max) & (lon >= lon_min) & (lon <= lon_max)
    return da.where(mask, drop=True)

### 4.2 Open fields (Z500/U/V) and inspect

In [17]:
# --- Open fields (explicit varnames from your files)
H = open_da(HGT_NC, "hgt")
U = open_da(U_NC,   "uwnd")
V = open_da(V_NC,   "vwnd")

print("Opened fields:")
print("H:", H.name, H.dims, H.shape)
print("U:", U.name, U.dims, U.shape)
print("V:", V.name, V.dims, V.shape)

# --- Debugging: check coordinates and dimensions of H
print("\nH dims:", H.dims)
print("H coords:", list(H.coords))
if "lat" in H.coords:
    print("lat dims/ndim:", H["lat"].dims, H["lat"].ndim)
if "lon" in H.coords:
    print("lon dims/ndim:", H["lon"].dims, H["lon"].ndim)

Opened fields:
H: hgt ('time', 'y', 'x') (4749, 100, 145)
U: uwnd ('time', 'y', 'x') (4749, 100, 145)
V: vwnd ('time', 'y', 'x') (4749, 100, 145)

H dims: ('time', 'y', 'x')
H coords: ['time', 'level', 'y', 'x', 'lat', 'lon']
lat dims/ndim: ('y', 'x') 2
lon dims/ndim: ('y', 'x') 2


### 4.3 Subset domain to Mexico-centered box

In [18]:
# --- Subset domain using the curvilinear lat/lon mask
H = subset_domain_curvilinear(H, LAT_MIN, LAT_MAX, LON_MIN, LON_MAX)
U = subset_domain_curvilinear(U, LAT_MIN, LAT_MAX, LON_MIN, LON_MAX)
V = subset_domain_curvilinear(V, LAT_MIN, LAT_MAX, LON_MIN, LON_MAX)

print("\nSubset fields:")
print("H:", H.dims, H.shape,
      f"lat[{float(H.lat.min()):.1f},{float(H.lat.max()):.1f}]",
      f"lon[{float(H.lon.min()):.1f},{float(H.lon.max()):.1f}]")


Subset fields:
H: ('time', 'y', 'x') (4749, 90, 140) lat[9.1,35.5] lon[-124.6,-78.1]


In [23]:
# --- 4.4b Mask diagnostics: % of cells inside the lat/lon box
# lat/lon are 2D (y,x) coords

lat = H["lat"]
lon = H["lon"]

mask = (lat >= LAT_MIN) & (lat <= LAT_MAX) & (lon >= LON_MIN) & (lon <= LON_MAX)

n_total = mask.size
n_in = int(mask.sum().values)
pct_in = 100.0 * n_in / n_total

print(f"Mask diagnostics: inside={n_in:,} / total={n_total:,}  ({pct_in:.1f}%)")

# Report lat/lon ranges of ONLY the masked (inside-box) cells
lat_in = lat.where(mask)
lon_in = lon.where(mask)

print("Inside-mask lat range:",
      float(lat_in.min(skipna=True)), "to", float(lat_in.max(skipna=True)))
print("Inside-mask lon range:",
      float(lon_in.min(skipna=True)), "to", float(lon_in.max(skipna=True)))

Mask diagnostics: inside=9,904 / total=12,600  (78.6%)
Inside-mask lat range: 12.003490447998047 to 32.99993896484375
Inside-mask lon range: -119.9977035522461 to -85.00567626953125


### 4.4 Align time coverage with PM tables (canonical = df_evt)

In [19]:
# Align time coverage with PM data (for consistency)
# Keep only overlapping dates across df_evt (canonical) and reanalysis time ranges.

pm_start, pm_end = df_evt.index.min(), df_evt.index.max()

# Convert xarray time to pandas timestamps for slicing
re_start = pd.to_datetime(H.time.values[0])
re_end   = pd.to_datetime(H.time.values[-1])

start = max(pm_start, re_start)
end   = min(pm_end, re_end)

print("Overlap window:", start.date(), "to", end.date())

# Slice PM tables to overlap
df_evt = df_evt.loc[start:end]
df_pm  = df_pm.loc[start:end]  # keep raw PM aligned too (useful for baseline plots)

# Slice reanalysis to overlap
H = H.sel(time=slice(np.datetime64(start), np.datetime64(end)))
U = U.sel(time=slice(np.datetime64(start), np.datetime64(end)))
V = V.sel(time=slice(np.datetime64(start), np.datetime64(end)))

print("PM days (df_evt):", df_evt.shape[0])
print("PM days (df_pm): ", df_pm.shape[0])
print("Reanalysis timesteps:", H.time.size)

Overlap window: 2012-01-01 to 2024-12-31
PM days (df_evt): 4555
PM days (df_pm):  4555
Reanalysis timesteps: 4749


### 4.5 Sanity check

In [20]:
# --- Final sanity checks: same time axis and same grid
if not (H.time.size == U.time.size == V.time.size):
    raise ValueError("Time dimension mismatch across H/U/V after slicing.")

if not (H.time.equals(U.time) and H.time.equals(V.time)):
    warnings.warn("H/U/V time coordinates are not identical (check reanalysis files).")

# Quick grid check
for coord in ["lat", "lon"]:
    if coord in H.coords and coord in U.coords and coord in V.coords:
        if not (H[coord].shape == U[coord].shape == V[coord].shape):
            warnings.warn(f"{coord} shape differs across H/U/V.")

print("OK: Reanalysis fields loaded, subset, and aligned to df_evt overlap window.")

OK: Reanalysis fields loaded, subset, and aligned to df_evt overlap window.


### 4.6 Time resolution diagnostics (reanalysis)

In [21]:
t = pd.to_datetime(H.time.values)
dt_counts = pd.Series(t).diff().value_counts().head(10)

print("Most common time steps (top 10):")
print(dt_counts)

print("\nFirst 5 timestamps:", t[:5])
print("Last  5 timestamps:", t[-5:])
print("Total timesteps:", len(t))

# how many timesteps per day (rough diagnostic)
ts_per_day = pd.Series(t).dt.floor("D").value_counts().sort_index()
print("\nTimesteps per day (value counts):")
print(ts_per_day.value_counts().head(10))

Most common time steps (top 10):
1 days    4748
Name: count, dtype: int64

First 5 timestamps: DatetimeIndex(['2012-01-01', '2012-01-02', '2012-01-03', '2012-01-04', '2012-01-05'], dtype='datetime64[ns]', freq=None)
Last  5 timestamps: DatetimeIndex(['2024-12-27', '2024-12-28', '2024-12-29', '2024-12-30', '2024-12-31'], dtype='datetime64[ns]', freq=None)
Total timesteps: 4749

Timesteps per day (value counts):
count
1    4749
Name: count, dtype: int64


## 5. Baseline meteorology (monthly climatology)

### 5.1 Sanity checks (dims, missing, ranges)

In [28]:
print("H dims:", H.dims, "shape:", H.shape)
print("U dims:", U.dims, "shape:", U.shape)
print("V dims:", V.dims, "shape:", V.shape)

print("\nTime coverage:")
print("H:", pd.to_datetime(H.time.values[0]).date(), "to", pd.to_datetime(H.time.values[-1]).date(), "| n=", H.time.size)
print("U:", pd.to_datetime(U.time.values[0]).date(), "to", pd.to_datetime(U.time.values[-1]).date(), "| n=", U.time.size)
print("V:", pd.to_datetime(V.time.values[0]).date(), "to", pd.to_datetime(V.time.values[-1]).date(), "| n=", V.time.size)

# Missing values (overall)
h_nan = int(H.isnull().sum().values)
u_nan = int(U.isnull().sum().values)
v_nan = int(V.isnull().sum().values)
print("\nTotal missing (all dims):")
print("H NaNs:", f"{h_nan:,}")
print("U NaNs:", f"{u_nan:,}")
print("V NaNs:", f"{v_nan:,}")

# Basic value ranges (global min/max; coarse but useful)
print("\nGlobal min/max (coarse check):")
print("H:", float(H.min().values), "to", float(H.max().values))
print("U:", float(U.min().values), "to", float(U.max().values))
print("V:", float(V.min().values), "to", float(V.max().values))

H dims: ('time', 'y', 'x') shape: (4749, 90, 140)
U dims: ('time', 'y', 'x') shape: (4749, 90, 140)
V dims: ('time', 'y', 'x') shape: (4749, 90, 140)

Time coverage:
H: 2012-01-01 to 2024-12-31 | n= 4749
U: 2012-01-01 to 2024-12-31 | n= 4749
V: 2012-01-01 to 2024-12-31 | n= 4749

Total missing (all dims):
H NaNs: 12,803,304
U NaNs: 12,803,304
V NaNs: 12,803,304

Global min/max (coarse check):
H: 5367.40478515625 to 6007.630859375
U: -30.42922019958496 to 54.217735290527344
V: -44.263675689697266 to 45.51524353027344


### 5.2 Monthly climatology (mean) — Z500, U, V

In [29]:
# --- mean fields
H_clim_mon = H.groupby("time.month").mean("time")
U_clim_mon = U.groupby("time.month").mean("time")
V_clim_mon = V.groupby("time.month").mean("time")

print("Monthly climatologies created:")
print("H_clim_mon:", H_clim_mon.dims, H_clim_mon.shape)
print("U_clim_mon:", U_clim_mon.dims, U_clim_mon.shape)
print("V_clim_mon:", V_clim_mon.dims, V_clim_mon.shape)

# check: months present
print("Months in H_clim_mon:", H_clim_mon["month"].values)
print("Months in U_clim_mon:", U_clim_mon["month"].values)
print("Months in V_clim_mon:", V_clim_mon["month"].values)

Monthly climatologies created:
H_clim_mon: ('month', 'y', 'x') (12, 90, 140)
U_clim_mon: ('month', 'y', 'x') (12, 90, 140)
V_clim_mon: ('month', 'y', 'x') (12, 90, 140)
Months in H_clim_mon: [ 1  2  3  4  5  6  7  8  9 10 11 12]
Months in U_clim_mon: [ 1  2  3  4  5  6  7  8  9 10 11 12]
Months in V_clim_mon: [ 1  2  3  4  5  6  7  8  9 10 11 12]


### 5.3 Monthly anomalies (Z500′); same definition as Stage 2

In [30]:
H_prime = H.groupby("time.month") - H_clim_mon

print("H_prime created:")
print("H_prime:", H_prime.dims, H_prime.shape)

# check: anomaly should be near-zero mean within each month
check = H_prime.groupby("time.month").mean("time")
max_abs_month_mean = float(np.abs(check).max().values)
print("Max abs(monthly-mean of H_prime):", max_abs_month_mean)

H_prime created:
H_prime: ('time', 'y', 'x') (4749, 90, 140)
Max abs(monthly-mean of H_prime): 0.007701071910560131


### 5.4 Monthly anomalies for winds; U′, V′ (fix prep)

In [31]:
# --- for consistency with H_prime in composites
U_prime = U.groupby("time.month") - U_clim_mon
V_prime = V.groupby("time.month") - V_clim_mon

print("U_prime / V_prime created:")
print("U_prime:", U_prime.dims, U_prime.shape)
print("V_prime:", V_prime.dims, V_prime.shape)

# check: anomaly month means should be near zero
u_check = U_prime.groupby("time.month").mean("time")
v_check = V_prime.groupby("time.month").mean("time")
print("Max abs(monthly-mean of U_prime):", float(np.abs(u_check).max().values))
print("Max abs(monthly-mean of V_prime):", float(np.abs(v_check).max().values))

U_prime / V_prime created:
U_prime: ('time', 'y', 'x') (4749, 90, 140)
V_prime: ('time', 'y', 'x') (4749, 90, 140)
Max abs(monthly-mean of U_prime): 1.693290141702164e-05
Max abs(monthly-mean of V_prime): 3.670868636618252e-06


### 5.5 Save baseline monthly means

In [32]:
out_nc = HERE / "outputs" / "data"
out_nc.mkdir(parents=True, exist_ok=True)
H_clim_mon.to_netcdf(out_nc / "stage3_H_clim_mon.nc")
U_clim_mon.to_netcdf(out_nc / "stage3_U_clim_mon.nc")
V_clim_mon.to_netcdf(out_nc / "stage3_V_clim_mon.nc")
print("saved: stage3 monthly climatologies (netcdf)")

saved: stage3 monthly climatologies (netcdf)


### 5.6 Z500′ distribution sanity checks (summary + histogram)

In [33]:
# --- last sanity check: Z500' anomaly distribution (domain-wide, all times)
hp_vals = H_prime.values
hp_mean = float(np.nanmean(hp_vals))
hp_std  = float(np.nanstd(hp_vals))
hp_p50  = float(np.nanpercentile(hp_vals, 50))
hp_p95  = float(np.nanpercentile(hp_vals, 95))
hp_p05  = float(np.nanpercentile(hp_vals, 5))

print(f"Z500′ summary (all times, domain-wide): mean={hp_mean:.3f}, std={hp_std:.3f}, "
      f"p05={hp_p05:.3f}, p50={hp_p50:.3f}, p95={hp_p95:.3f}")

# Histogram diagnostic (reproducible via RANDOM_SEED)
sample = hp_vals.ravel()
sample = sample[np.isfinite(sample)]
if sample.size > 0:
    rng = np.random.default_rng(RANDOM_SEED)
    sample = rng.choice(sample, size=min(200_000, sample.size), replace=False)

    plt.figure(figsize=(7, 4), dpi=140)
    plt.hist(sample, bins=80)
    plt.title("Z500′ anomaly distribution (monthly baseline; sample)")
    plt.xlabel("Z500′ (meters)")
    plt.ylabel("Count")
    out = OUT_FIG / "stage3_z500_anomaly_hist_monthly_baseline.png"
    plt.tight_layout()
    plt.savefig(out, dpi=200)
    plt.close()
    print(f"saved: {out}")
else:
    print("No finite Z500′ values found for histogram check.")

Z500′ summary (all times, domain-wide): mean=0.000, std=31.484, p05=-48.815, p50=1.012, p95=46.859
saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 3\outputs\figures\stage3_z500_anomaly_hist_monthly_baseline.png


## 6. Downloading precipitation (daily) + baseline precip climatology

- Precipitation was added per Prof. Chu recommendation to anchor the monthly meteorological climatology (wet vs dry season context).  
- The first part revolves around downloading NARR daily precipitation **rate** (`prate`) via direct HTTP (not OPeNDAP), verify the signal year-by-year, convert to **mm/day**, crop to the Mexico-centered domain, and save a single merged NetCDF for 2012–2024.
- The second part is focused on building the precipitacion climatology.

In [61]:
# from pathlib import Path
# import gc
# import os

# # Archivo objetivo
# target = PARENT / "precip_sfc_mex_2012_2024.nc"

# # Cerrar posibles datasets abiertos en el kernel
# for name in ["ds", "ds0", "ds_p", "remote"]:
#     obj = globals().get(name, None)
#     if obj is not None and hasattr(obj, "close"):
#         try:
#             obj.close()
#             print(f"Cerrado: {name}")
#         except Exception as e:
#             print(f"No se pudo cerrar {name}: {e}")

# gc.collect()

# # Borrar archivo
# if target.exists():
#     try:
#         os.remove(target)
#         print(f"Eliminado: {target}")
#     except PermissionError as e:
#         print("No se pudo eliminar (archivo en uso).")
#         print("Sugerencia: reinicia el kernel y vuelve a ejecutar esta celda.")
#         print(e)
# else:
#     print(f"No existe: {target}")

No se pudo cerrar ds: NetCDF: Not a valid ID
Cerrado: ds0
Cerrado: ds_p
Cerrado: remote
Eliminado: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\precip_sfc_mex_2012_2024.nc


### 6.1 Paths + raw folder

In [56]:
# --- Paths for raw downloads and final merged output
from urllib.request import urlretrieve

BASE_DL = "https://downloads.psl.noaa.gov/Datasets/NARR/Dailies/monolevel/"
YEARS = range(2012, 2025)  # 2012–2024
RAW_DIR = HERE / "outputs" / "data" / "precip_raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

OUT_PRECIP = PARENT / "precip_sfc_mex_2012_2024.nc"

print("raw dir:", RAW_DIR)
print("final out:", OUT_PRECIP)

raw dir: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 3\outputs\data\precip_raw
final out: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\precip_sfc_mex_2012_2024.nc


### 6.2 Download prate.YYYY.nc files (direct HTTP)

In [57]:
# --- Download yearly prate files
raw_files = []
for y in YEARS:
    fn = f"prate.{y}.nc"
    url = BASE_DL + fn
    local = RAW_DIR / fn
    raw_files.append(local)

    if local.exists() and local.stat().st_size > 0:
        print(f"{y}: exists ({local.stat().st_size/1e6:.1f} MB)")
        continue

    print(f"{y}: downloading...")
    urlretrieve(url, local)
    print(f"{y}: saved ({local.stat().st_size/1e6:.1f} MB)")

2012: downloading...
2012: saved (65.9 MB)
2013: downloading...
2013: saved (68.1 MB)
2014: downloading...
2014: saved (68.2 MB)
2015: downloading...
2015: saved (64.1 MB)
2016: downloading...
2016: saved (63.4 MB)
2017: downloading...
2017: saved (63.0 MB)
2018: downloading...
2018: saved (64.7 MB)
2019: downloading...
2019: saved (64.4 MB)
2020: downloading...
2020: saved (64.4 MB)
2021: downloading...
2021: saved (65.1 MB)
2022: downloading...
2022: saved (67.1 MB)
2023: downloading...
2023: saved (67.1 MB)
2024: downloading...
2024: saved (67.7 MB)


### 6.3 Verify signal per year (bbox max + JJAS mean) + build merged file (mm/day, cropped)

In [58]:
# --- Verify (per-year) + build merged precip (mm/day) cropped to Mexico domain
def _standardize_lon(lon_da: xr.DataArray) -> xr.DataArray:
    if float(lon_da.max()) > 180.0:
        return (((lon_da + 180) % 360) - 180)
    return lon_da

def _detect_bbox_indices(ds: xr.Dataset) -> tuple[int, int, int, int]:
    lat2d = ds["lat"] if "lat" in ds.variables else ds.coords["lat"]
    lon2d = ds["lon"] if "lon" in ds.variables else ds.coords["lon"]
    lon2d = _standardize_lon(lon2d)

    m = (lon2d >= LON_MIN) & (lon2d <= LON_MAX) & (lat2d >= LAT_MIN) & (lat2d <= LAT_MAX)
    yy, xx = np.where(m.values)
    if yy.size == 0 or xx.size == 0:
        raise RuntimeError("Mask produced zero cells (check LAT/LON bounds).")
    return int(yy.min()), int(yy.max()), int(xx.min()), int(xx.max())

In [62]:
# detect bbox indices once (use first year)
ds0 = xr.open_dataset(raw_files[0])
y0, y1, x0, x1 = _detect_bbox_indices(ds0)
ds0.close()
print(f"bbox indices: y={y0}..{y1} (n={y1-y0+1}), x={x0}..{x1} (n={x1-x0+1})")

pieces = []
print("\nYearly checks (cropped bbox, prate->mm/day):")
for f in raw_files:
    y = int(f.name.split(".")[1])

    ds = xr.open_dataset(f)
    if "prate" not in ds.data_vars:
        ds.close()
        raise KeyError(f"'prate' not found in {f.name}. data_vars={list(ds.data_vars)}")

    pr = ds["prate"].sortby("time")

    # standardize lon for consistency (masking/plots later)
    if "lon" in pr.coords:
        pr = pr.assign_coords(lon=_standardize_lon(pr["lon"]))

    pr = pr.isel(y=slice(y0, y1+1), x=slice(x0, x1+1))

    # convert to mm/day
    pr_mmday = pr * 86400.0
    pr_mmday.name = "precip_mmday"
    pr_mmday.attrs["units"] = "mm/day"
    pr_mmday.attrs["source_var"] = "prate"
    pr_mmday.attrs["source_units"] = ds["prate"].attrs.get("units", "unknown")

    # verification stats
    vmax = float(pr_mmday.max(skipna=True).values)
    jjas = pr_mmday.sel(time=pr_mmday["time"].dt.month.isin([6, 7, 8, 9]))
    jjas_mean = float(jjas.mean(skipna=True).values) if jjas.time.size else np.nan
    print(f"{y}: bbox max={vmax:.2f} mm/day | JJAS mean={jjas_mean:.3f} mm/day")

    # modest chunks, append
    pr_mmday = pr_mmday.chunk({"time": min(pr_mmday.sizes["time"], 366)})
    pieces.append(pr_mmday)

    ds.close()

# concat + sort + write
P = xr.concat(pieces, dim="time", compat="override", join="override", coords="minimal").sortby("time")

comp = dict(zlib=True, complevel=3)
P.to_netcdf(OUT_PRECIP, encoding={P.name: comp})
print(f"\nsaved: {OUT_PRECIP}  ({P.sizes['time']} days)")

bbox indices: y=3..92 (n=90), x=122..261 (n=140)

Yearly checks (cropped bbox, prate->mm/day):
2012: bbox max=245.95 mm/day | JJAS mean=4.003 mm/day
2013: bbox max=241.13 mm/day | JJAS mean=3.968 mm/day
2014: bbox max=256.65 mm/day | JJAS mean=3.954 mm/day
2015: bbox max=331.51 mm/day | JJAS mean=4.132 mm/day
2016: bbox max=295.31 mm/day | JJAS mean=4.011 mm/day
2017: bbox max=268.89 mm/day | JJAS mean=4.085 mm/day
2018: bbox max=295.44 mm/day | JJAS mean=3.800 mm/day
2019: bbox max=270.78 mm/day | JJAS mean=3.448 mm/day
2020: bbox max=306.12 mm/day | JJAS mean=3.875 mm/day
2021: bbox max=265.54 mm/day | JJAS mean=4.258 mm/day
2022: bbox max=266.14 mm/day | JJAS mean=3.767 mm/day
2023: bbox max=298.71 mm/day | JJAS mean=3.015 mm/day
2024: bbox max=316.37 mm/day | JJAS mean=3.874 mm/day

saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\precip_sfc_mex_2012_2024.nc  (4749 days)


### 6.4 Open merged precip file + sanity checks

In [63]:
dsP = xr.open_dataset(OUT_PRECIP)
print(dsP)

P = dsP["precip_mmday"].sortby("time")

print("\nPrecip variable:", P.name)
print("dims:", P.dims, "shape:", P.shape)
print("time:", pd.to_datetime(P.time.values[0]).date(), "to", pd.to_datetime(P.time.values[-1]).date(), "| n=", P.time.size)
print("units:", P.attrs.get("units", "unknown"))

print("\nGlobal min/max:", float(P.min(skipna=True)), "to", float(P.max(skipna=True)))
print("Total NaNs:", int(P.isnull().sum().values))

<xarray.Dataset> Size: 239MB
Dimensions:       (time: 4749, y: 90, x: 140)
Coordinates:
  * time          (time) datetime64[ns] 38kB 2012-01-01 ... 2024-12-31
  * y             (y) float32 360B 9.739e+04 1.299e+05 ... 2.954e+06 2.987e+06
  * x             (x) float32 560B 3.96e+06 3.993e+06 ... 8.44e+06 8.473e+06
    lat           (y, x) float32 50kB ...
    lon           (y, x) float32 50kB ...
Data variables:
    precip_mmday  (time, y, x) float32 239MB ...

Precip variable: precip_mmday
dims: ('time', 'y', 'x') shape: (4749, 90, 140)
time: 2012-01-01 to 2024-12-31 | n= 4749
units: mm/day

Global min/max: -1.1310551166534424 to 331.51068115234375
Total NaNs: 0


### 6.5 Align precip to df_evt overlap (consistency)

In [64]:
start, end = df_evt.index.min(), df_evt.index.max()

P = P.sel(time=slice(np.datetime64(start), np.datetime64(end)))

print("Aligned precip to:", start.date(), "to", end.date(), "| n=", P.time.size)

Aligned precip to: 2012-01-01 to 2024-12-31 | n= 4749


### 6.6 Monthly precipitation climatology (12-panel maps)

In [72]:
# --- Compute Valley of Mexico regional precipitation climatology
lat2d = P["lat"]
lon2d = P["lon"]

vom_mask = (
    (lon2d >= SW_lon) & (lon2d <= NE_lon) &
    (lat2d >= SW_lat) & (lat2d <= NE_lat)
)

P_vom_daily = P.where(vom_mask).mean(dim=("y", "x"), skipna=True)
P_vom_clim = P_vom_daily.groupby("time.month").mean("time", skipna=True)

print("Valley of Mexico monthly climatology computed:")
print(P_vom_clim.values)

Valley of Mexico monthly climatology computed:
[0.30730453 0.3094115  0.6884464  0.79704511 1.91232312 4.38016415
 4.63218594 4.97546005 5.16063643 2.44687152 0.95886058 0.38728735]


In [76]:
# --- monthly precip climatology maps (12-panel; mm/day)
P_clim_mon = P.groupby("time.month").mean("time", skipna=True)

# color scale (robust)
vmax = float(np.nanpercentile(P_clim_mon.values, 99))
if (not np.isfinite(vmax)) or vmax <= 0:
    vmax = float(np.nanmax(P_clim_mon.values))

proj = ccrs.PlateCarree()
fig, axes = plt.subplots(3, 4, figsize=(16, 9), dpi=250,
                         subplot_kw={"projection": proj})
axes = axes.flatten()

lon2d = P_clim_mon["lon"].values
lat2d = P_clim_mon["lat"].values

pcm_ref = None

for i, m in enumerate(range(1, 13)):
    ax = axes[i]
    ax.set_extent([LON_MIN, LON_MAX, LAT_MIN, LAT_MAX], crs=proj)

    ax.coastlines(resolution="50m", linewidth=0.5)
    ax.add_feature(cfeature.BORDERS, linewidth=0.4)
    ax.add_feature(cfeature.STATES.with_scale("50m"), linewidth=0.3)

    pm = P_clim_mon.sel(month=m)

    pcm = ax.pcolormesh(lon2d, lat2d, pm.values,
                        cmap="viridis", shading="auto",
                        vmin=0, vmax=vmax, transform=proj)
    pcm_ref = pcm

    # Valley of Mexico box + CDMX star
    rect = mpatches.Rectangle((VOM_BOX[0], VOM_BOX[1]),
                              VOM_BOX[2], VOM_BOX[3],
                              fill=False, edgecolor="k",
                              linewidth=1, transform=proj)
    ax.add_patch(rect)

    ax.plot(LON_CDMX, LAT_CDMX, marker="*", color="gold",
            markersize=8, markeredgecolor="k", transform=proj)

    # Month title + precipitation in VOM
    precip_vom = float(P_vom_clim.sel(month=m).values)
    ax.set_title(f"{calendar.month_abbr[m]} ({precip_vom:.2f} mm/day)", 
                 fontsize=10, weight="bold")

    # Grid labels: only left column + bottom row
    gl = ax.gridlines(draw_labels=True, linewidth=0.2, color="gray",
                      alpha=0.5, linestyle="--")
    gl.top_labels = False
    gl.right_labels = False
    if (i % 4) != 0:
        gl.left_labels = False
    if i < 8:
        gl.bottom_labels = False

# Colorbar (right side)
cbar_ax = fig.add_axes([0.92, 0.20, 0.015, 0.60])
cb = fig.colorbar(pcm_ref, cax=cbar_ax)
cb.set_label("Precipitation (mm/day)")

# Titles
fig.suptitle("NARR Monthly precipitation climatology (2012–2024)",
             fontsize=18, weight="bold", y=0.96)
fig.text(0.5, 0.91,
         "12-panel spatial maps | 'prate' → mm/day",
         ha="center", fontsize=14, style="italic")

# Legend elements
legend_elements = [
    mpatches.Rectangle((0, 0), 1, 1, fill=False, edgecolor="k", linewidth=1.5,
                       label="Valley of Mexico"),
    plt.Line2D([0], [0], marker="*", color="w", markerfacecolor="gold",
              markeredgecolor="k", markersize=12, label="Mexico City (CDMX)")
]
fig.legend(handles=legend_elements, loc="upper left", fontsize=11,
          frameon=False, bbox_to_anchor=(0.01, 0.97))

plt.tight_layout(rect=[0, 0, 0.9, 0.93])
out = OUT_FIG / "stage3_precip_climatology_monthly_12panel_mmday.png"
plt.savefig(out, dpi=300, bbox_inches="tight")
plt.close(fig)
print(f"saved: {out}")

saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 3\outputs\figures\stage3_precip_climatology_monthly_12panel_mmday.png


### 6.7 Regional climatology over Valley of Mexico (monthly cycle)

In [78]:
# --- Valley of Mexico regional monthly climatology (mm/day)
lat2d = P["lat"]
lon2d = P["lon"]

vom_mask = (
    (lon2d >= SW_lon) & (lon2d <= NE_lon) &
    (lat2d >= SW_lat) & (lat2d <= NE_lat)
)

n_true = int(vom_mask.sum().values)
print(f"VOM mask cells inside: {n_true}")

P_vom_daily = P.where(vom_mask).mean(dim=("y", "x"), skipna=True)
P_vom_clim = P_vom_daily.groupby("time.month").mean("time", skipna=True)

# Enhanced timeseries plot
fig, ax = plt.subplots(figsize=(10, 5), dpi=250)

# Highlight wet season (Jun-Oct) with background shading
ax.axvspan(6, 10, alpha=0.1, color='blue', label='Wet season (Jun–Oct)')

# Annual mean line
annual_mean = float(P_vom_clim.mean())
ax.axhline(annual_mean, color='gray', linestyle='--', linewidth=1, alpha=0.7, 
           label=f'Annual mean: {annual_mean:.2f} mm/day')

# Main line plot with enhanced styling
months = np.arange(1, 13)
ax.plot(months, P_vom_clim.values, marker='o', markersize=8, 
        linewidth=2.5, color='#1f77b4', markeredgecolor='white', 
        markeredgewidth=1.5, label='Monthly climatology')

# Add value labels on each point
for m, val in zip(months, P_vom_clim.values):
    ax.text(m, val + 0.15, f'{val:.2f}', ha='center', va='bottom', 
            fontsize=9, weight='bold')

# Styling
ax.set_xticks(months)
ax.set_xticklabels([calendar.month_abbr[m] for m in months])
ax.set_xlabel("Month", fontsize=12, weight='bold')
ax.set_ylabel("Precipitation (mm/day)", fontsize=12, weight='bold')
ax.set_title("Valley of Mexico precipitation climatology (2012–2024)", 
             fontsize=14, weight='bold', pad=15)
ax.grid(True, alpha=0.3, linestyle='--')
ax.legend(loc='upper left', fontsize=10, framealpha=0.95)

# Set y-axis limits with some padding
ymax = P_vom_clim.values.max()
ax.set_ylim(0, ymax * 1.15)

out = OUT_FIG / "stage3_precip_climatology_vom_timeseries_mmday.png"
plt.tight_layout()
plt.savefig(out, dpi=250, bbox_inches='tight')
plt.close()
print(f"saved: {out}")

VOM mask cells inside: 120
saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 3\outputs\figures\stage3_precip_climatology_vom_timeseries_mmday.png


### 6.8 Save monthly climatology table for slides (VOM)

In [67]:
# --- Export VOM monthly climatology to CSV (for slides/reporting)
vom_tab = pd.DataFrame({
    "month": np.arange(1, 13),
    "month_name": [calendar.month_name[m] for m in range(1, 13)],
    "precip_mmday": P_vom_clim.values
})

out = OUT_TAB / "stage3_precip_climatology_vom_mmday.csv"
vom_tab.to_csv(out, index=False)
print(f"saved: {out}")

vom_tab

saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 3\outputs\tables\stage3_precip_climatology_vom_mmday.csv


,month,month_name,precip_mmday
0,1,January,0.31
1,2,February,0.31
2,3,March,0.69
3,4,April,0.80
4,5,May,1.91
5,6,June,4.38
6,7,July,4.63
7,8,August,4.98
8,9,September,5.16
9,10,October,2.45


## 7. Defining p10 and enforcing comparability (N control)

- Defining low-pollution days (p10) by calendar month, consistent with the Stage 2 p90 logic.  
- Computing monthly sample sizes for high vs low composites and define `N_used = min(N_high, N_low)` to enforce comparability in the monthly maps.

### 7.1 Compute monthly p10 thresholds and flags (PM10, PM2.5)

In [80]:
# --- Computing thresholds by calendar month (ignoring NaNs) + attaching to df_evt (robust)
# df_evt is canonical daily table with a DatetimeIndex and a 'month' column

# monthly p10 thresholds (Series indexed by month 1..12)
thr_p10_pm10 = df_evt.groupby("month")["PM10"].quantile(0.10)
thr_p10_pm25 = df_evt.groupby("month")["PM2.5"].quantile(0.10)

# attach thresholds to each day via month->threshold mapping
df_evt["thr_p10_PM10"]  = df_evt["month"].map(thr_p10_pm10)
df_evt["thr_p10_PM2.5"] = df_evt["month"].map(thr_p10_pm25)

# Create p10 event flags (1 = low day)
df_evt["evt_PM10_p10"]  = (df_evt["PM10"]  <= df_evt["thr_p10_PM10"]).astype(int)
df_evt["evt_PM2.5_p10"] = (df_evt["PM2.5"] <= df_evt["thr_p10_PM2.5"]).astype(int)

# Build the p10 threshold table for saving/reporting
thr_p10_df = pd.DataFrame({
    "month": np.arange(1, 13),
    "month_name": [calendar.month_name[m] for m in range(1, 13)],
    "thr_p10_PM10":  [float(thr_p10_pm10.loc[m]) for m in range(1, 13)],
    "thr_p10_PM2.5": [float(thr_p10_pm25.loc[m]) for m in range(1, 13)],
})

print("Computed p10 thresholds + flags (by calendar month).")
print(thr_p10_df.to_string(index=False))

Computed p10 thresholds + flags (by calendar month).
 month month_name  thr_p10_PM10  thr_p10_PM2.5
     1    January         34.33          14.13
     2   February         35.86          14.30
     3      March         36.00          15.43
     4      April         35.43          17.33
     5        May         31.33          18.04
     6       June         19.45          10.62
     7       July         22.50          12.50
     8     August         20.60          10.37
     9  September         19.32           9.56
    10    October         20.20           8.84
    11   November         27.86          13.36
    12   December         34.24          16.26


### 7.2 Sanity checks (p10 flags, counts by month)

In [ ]:
# --- p10 should be ~10% of available days per month (per pollutant)
def _monthly_flag_share(df, pol, flag):
    out = []
    for m in range(1, 13):
        sub = df[df["month"] == m]
        n_avail = sub[pol].notna().sum()
        n_flag = sub.loc[sub[pol].notna(), flag].sum()
        share = (n_flag / n_avail) if n_avail > 0 else np.nan
        out.append([m, n_avail, int(n_flag), share])
    return pd.DataFrame(out, columns=["month", "n_available", "n_flag", "share"])

check_pm10 = _monthly_flag_share(df_evt.reset_index(), "PM10", "evt_PM10_p10")
check_pm25 = _monthly_flag_share(df_evt.reset_index(), "PM2.5", "evt_PM2.5_p10")

print("PM10 p10 share by month (should be near 0.10):")
print(check_pm10.to_string(index=False))

print("\nPM2.5 p10 share by month (should be near 0.10):")
print(check_pm25.to_string(index=False))

PM10 p10 share by month (should be near 0.10):
 month  n_available  n_flag  share
     1          373      38   0.10
     2          353      36   0.10
     3          401      43   0.11
     4          381      39   0.10
     5          372      40   0.11
     6          360      36   0.10
     7          398      42   0.11
     8          396      41   0.10
     9          370      37   0.10
    10          377      40   0.11
    11          374      38   0.10
    12          400      40   0.10

PM2.5 p10 share by month (should be near 0.10):
 month  n_available  n_flag  share
     1          373      38   0.10
     2          353      36   0.10
     3          401      41   0.10
     4          381      39   0.10
     5          372      38   0.10
     6          360      36   0.10
     7          398      43   0.11
     8          396      40   0.10
     9          370      37   0.10
    10          377      38   0.10
    11          374      38   0.10
    12          400      40  

### 7.3 Build monthly N table (N_high, N_low, N_used) for p90 vs p10

- Key “comparability control” table.

In [82]:
# --- Monthly N control table for composites (p90 vs p10)
def build_n_control(df: pd.DataFrame, pol: str) -> pd.DataFrame:
    if pol == "PM10":
        flag_hi = "evt_PM10_p90"
        flag_lo = "evt_PM10_p10"
    elif pol == "PM2.5":
        flag_hi = "evt_PM2.5_p90"
        flag_lo = "evt_PM2.5_p10"
    else:
        raise ValueError("pol must be 'PM10' or 'PM2.5'")

    rows = []
    for m in range(1, 13):
        sub = df[df["month"] == m]
        n_avail = sub[pol].notna().sum()

        n_high = sub.loc[sub[pol].notna(), flag_hi].sum()
        n_low  = sub.loc[sub[pol].notna(), flag_lo].sum()
        n_used = int(min(n_high, n_low))

        rows.append([m, int(n_avail), int(n_high), int(n_low), n_used])

    out = pd.DataFrame(rows, columns=["month", "n_available", "N_high_p90", "N_low_p10", "N_used"])
    out["month_name"] = out["month"].apply(lambda m: calendar.month_name[m])
    out = out[["month", "month_name", "n_available", "N_high_p90", "N_low_p10", "N_used"]]
    return out

nctrl_pm10 = build_n_control(df_evt, "PM10")
nctrl_pm25 = build_n_control(df_evt, "PM2.5")

print("N-control table (PM10):")
print(nctrl_pm10.to_string(index=False))
print("\nN-control table (PM2.5):")
print(nctrl_pm25.to_string(index=False))

N-control table (PM10):
 month month_name  n_available  N_high_p90  N_low_p10  N_used
     1    January          373          38         38      38
     2   February          353          36         36      36
     3      March          401          41         43      41
     4      April          381          39         39      39
     5        May          372          38         40      38
     6       June          360          36         36      36
     7       July          398          40         42      40
     8     August          396          41         41      41
     9  September          370          37         37      37
    10    October          377          39         40      39
    11   November          374          38         38      38
    12   December          400          41         40      40

N-control table (PM2.5):
 month month_name  n_available  N_high_p90  N_low_p10  N_used
     1    January          373          39         38      38
     2   February   

### 7.4 Save outputs (threshold table + N-control tables)

In [83]:
out_thr = OUT_TAB / "stage3_p10_thresholds_by_calendar_month.csv"
thr_p10_df.to_csv(out_thr, index=False)
print(f"saved: {out_thr}")

out_n10 = OUT_TAB / "stage3_N_control_p90_vs_p10_PM10.csv"
out_n25 = OUT_TAB / "stage3_N_control_p90_vs_p10_PM25.csv"

nctrl_pm10.to_csv(out_n10, index=False)
nctrl_pm25.to_csv(out_n25, index=False)

print(f"saved: {out_n10}")
print(f"saved: {out_n25}")

saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 3\outputs\tables\stage3_p10_thresholds_by_calendar_month.csv
saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 3\outputs\tables\stage3_N_control_p90_vs_p10_PM10.csv
saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 3\outputs\tables\stage3_N_control_p90_vs_p10_PM25.csv


## 8. Monthly composites (p90, p10, and p90-p10)

Computing 12-month composites of Z500′ and 500 hPa wind anomalies (U′, V′) for:
- p90 (high-PM) days
- p10 (low-PM) days
- difference (p90 - p10), using balanced sample sizes per month (`N_used`).
All figures are saved to `outputs/figures/`.

### 8.1 Helper functions (balanced date selection + composites)

In [84]:
# --- helpers: balanced (N_used) date selection + composite mean (skipna)
def composite_mean(da: xr.DataArray, dates: pd.DatetimeIndex) -> xr.DataArray:
    """Mean composite over provided dates (skipna=True)."""
    if len(dates) == 0:
        return da.isel(time=0) * np.nan
    return da.sel(time=dates).mean("time", skipna=True)


def _n_used_for_month(nctrl: pd.DataFrame, m: int) -> int:
    """Fetch N_used for month m from an N-control table."""
    return int(nctrl.loc[nctrl["month"] == m, "N_used"].values[0])


def balanced_extreme_dates(df: pd.DataFrame, pol: str, m: int, kind: str, n_used: int) -> pd.DatetimeIndex:
    """
    Deterministic selection of exactly n_used days for a given month and extreme kind:
      kind = "p90" (high) or "p10" (low)

    Uses existing flags (evt_*), then ranks by concentration (severity) and breaks ties by DATE.
    """
    if pol == "PM10":
        flag_hi, flag_lo = "evt_PM10_p90", "evt_PM10_p10"
    elif pol == "PM2.5":
        flag_hi, flag_lo = "evt_PM2.5_p90", "evt_PM2.5_p10"
    else:
        raise ValueError("pol must be 'PM10' or 'PM2.5'")

    if kind == "p90":
        flag = flag_hi
        asc = False  # highest first
    elif kind == "p10":
        flag = flag_lo
        asc = True   # lowest first
    else:
        raise ValueError("kind must be 'p90' or 'p10'")

    sub = df[(df["month"] == m) & (df[flag] == 1) & (df[pol].notna())].copy()

    if sub.empty or n_used <= 0:
        return pd.DatetimeIndex([])

    # rank by concentration (severity), then by date to break ties deterministically
    sub = sub.sort_values([pol], ascending=asc)
    sub = sub.sort_index()  # DATE as secondary tie-breaker

    # for p90 we want largest values -> reverse after date-tie break
    if kind == "p90":
        sub = sub.sort_values([pol], ascending=False).sort_index()

    return pd.DatetimeIndex(sub.index[:n_used])

### 8.2 Precompute monthly composites (p90, p10, diff) + global color scales

In [ ]:
# --- precomputing monthly composites for one pollutant (balanced N_used)
def compute_monthly_composites_balanced(pol: str, nctrl: pd.DataFrame, label: str):
    """
    Compute balanced monthly composites (p90, p10, and diff) for one pollutant.
    Returns dicts keyed by month for:
      Hp90, Up90, Vp90
      Hp10, Up10, Vp10
      Hdif, Udif, Vdif
    plus max_abs scales for (p90, p10, dif).
    """
    Hp90, Up90, Vp90 = {}, {}, {}
    Hp10, Up10, Vp10 = {}, {}, {}
    Hdif, Udif, Vdif = {}, {}, {}

    maxabs_p90 = 0.0
    maxabs_p10 = 0.0
    maxabs_dif = 0.0

    for m in range(1, 13):
        n_used = _n_used_for_month(nctrl, m)

        d_hi = balanced_extreme_dates(df_evt, pol, m, "p90", n_used)
        d_lo = balanced_extreme_dates(df_evt, pol, m, "p10", n_used)

        # composites on anomalies: H' with U'/V'
        hp_hi = composite_mean(H_prime, d_hi)
        up_hi = composite_mean(U_prime, d_hi)
        vp_hi = composite_mean(V_prime, d_hi)

        hp_lo = composite_mean(H_prime, d_lo)
        up_lo = composite_mean(U_prime, d_lo)
        vp_lo = composite_mean(V_prime, d_lo)

        hp_d = hp_hi - hp_lo
        up_d = up_hi - up_lo
        vp_d = vp_hi - vp_lo

        Hp90[m], Up90[m], Vp90[m] = hp_hi, up_hi, vp_hi
        Hp10[m], Up10[m], Vp10[m] = hp_lo, up_lo, vp_lo
        Hdif[m], Udif[m], Vdif[m] = hp_d, up_d, vp_d

        # global maxabs scales
        a90 = float(np.nanmax(np.abs(hp_hi.values)))
        a10 = float(np.nanmax(np.abs(hp_lo.values)))
        adf = float(np.nanmax(np.abs(hp_d.values)))

        if np.isfinite(a90): maxabs_p90 = max(maxabs_p90, a90)
        if np.isfinite(a10): maxabs_p10 = max(maxabs_p10, a10)
        if np.isfinite(adf): maxabs_dif = max(maxabs_dif, adf)

        print(f"{label} | month={m:02d} | N_used={n_used} | max|Hp90|={a90:.2f} | max|Hp10|={a10:.2f} | max|Hdiff|={adf:.2f}")

    return (Hp90, Up90, Vp90, maxabs_p90,
            Hp10, Up10, Vp10, maxabs_p10,
            Hdif, Udif, Vdif, maxabs_dif)

In [ ]:
# Compute for both pollutants (balanced using the Section 7 N-control tables)
Hp90_pm25, Up90_pm25, Vp90_pm25, max90_pm25, Hp10_pm25, Up10_pm25, Vp10_pm25, max10_pm25, Hdif_pm25, Udif_pm25, Vdif_pm25, maxdf_pm25 = \
    compute_monthly_composites_balanced("PM2.5", nctrl_pm25, "PM2.5")

Hp90_pm10, Up90_pm10, Vp90_pm10, max90_pm10, Hp10_pm10, Up10_pm10, Vp10_pm10, max10_pm10, Hdif_pm10, Udif_pm10, Vdif_pm10, maxdf_pm10 = \
    compute_monthly_composites_balanced("PM10", nctrl_pm10, "PM10")

print("\nGlobal maxabs (PM2.5): p90", max90_pm25, "| p10", max10_pm25, "| diff", maxdf_pm25)
print("Global maxabs (PM10):  p90", max90_pm10, "| p10", max10_pm10, "| diff", maxdf_pm10)

PM2.5 | month=01 | N_used=38 | max|Hp90|=14.82 | max|Hp10|=63.73 | max|Hdiff|=75.58
PM2.5 | month=02 | N_used=36 | max|Hp90|=34.91 | max|Hp10|=70.11 | max|Hdiff|=99.95
PM2.5 | month=03 | N_used=41 | max|Hp90|=17.64 | max|Hp10|=58.34 | max|Hdiff|=58.49
PM2.5 | month=04 | N_used=39 | max|Hp90|=15.25 | max|Hp10|=39.13 | max|Hdiff|=45.55
PM2.5 | month=05 | N_used=38 | max|Hp90|=56.28 | max|Hp10|=38.10 | max|Hdiff|=70.63
PM2.5 | month=06 | N_used=36 | max|Hp90|=18.53 | max|Hp10|=27.47 | max|Hdiff|=42.43
PM2.5 | month=07 | N_used=40 | max|Hp90|=15.11 | max|Hp10|=16.02 | max|Hdiff|=29.48
PM2.5 | month=08 | N_used=40 | max|Hp90|=18.80 | max|Hp10|=10.93 | max|Hdiff|=22.58
PM2.5 | month=09 | N_used=37 | max|Hp90|=15.90 | max|Hp10|=13.33 | max|Hdiff|=28.01
PM2.5 | month=10 | N_used=38 | max|Hp90|=45.65 | max|Hp10|=20.31 | max|Hdiff|=64.77
PM2.5 | month=11 | N_used=38 | max|Hp90|=26.49 | max|Hp10|=47.29 | max|Hdiff|=69.45
PM2.5 | month=12 | N_used=40 | max|Hp90|=25.98 | max|Hp10|=36.57 | max|Hdiff

### 8.3 Plotting (12-panel multipanel; vectors = U′/V′)

Same as `plot_monthly_multipanel` de Stage 2, only with 2 small changes: 
- `title_main`/ `subtitle` para reusar en p90/p10/diff, 
- “Saved:” → “saved:”. 
Besides, stippling stays off if `P_month` has NaNs; later on in Section 9 significance will be calculated.

In [107]:
def _nan_pvals_like(hp_dict: dict) -> dict:
    """P_month placeholder: all-NaN arrays to disable stippling."""
    out = {}
    for m, da in hp_dict.items():
        p = da * np.nan
        out[m] = p
    return out

def plot_monthly_multipanel(label: str,
                            Hp_month: dict,
                            U_month: dict,
                            V_month: dict,
                            P_month: dict,
                            max_abs: float,
                            outpath: Path,
                            title_main: str,
                            subtitle: str,
                            nctrl: pd.DataFrame,
                            frac_area_dict: dict = None,
                            p_thresh: float = 0.05,
                            vector_step: int = 6,
                            stipple_thin: int = 8):
    """
    Stage-2 style multipanel plot (3x4).
    Shading: Z500′ anomaly (TwoSlopeNorm centered at 0)
    Contours: black (pos solid / neg dashed)
    Vectors: black (U′/V′ here)
    Stippling: p<0.05 (disabled if P_month is all NaNs)
    + Valley of Mexico box + CDMX star
    + N_used displayed in each month title
    + For section 9: frac_area (fraction of area with p<0.05) also in title
    """
    proj = ccrs.PlateCarree()
    fig, axes = plt.subplots(3, 4, figsize=(16, 9), dpi=250, subplot_kw={"projection": proj})
    axes = axes.flatten()

    lon2d = H_prime["lon"].values
    lat2d = H_prime["lat"].values

    norm = TwoSlopeNorm(vcenter=0, vmin=-max_abs, vmax=max_abs)
    pcm_ref = None

    for i, m in enumerate(range(1, 13)):
        ax = axes[i]
        ax.set_extent([LON_MIN, LON_MAX, LAT_MIN, LAT_MAX], crs=proj)

        ax.coastlines(resolution="50m", linewidth=0.5)
        ax.add_feature(cfeature.BORDERS, linewidth=0.4)
        ax.add_feature(cfeature.STATES.with_scale("50m"), linewidth=0.3)

        Hp = Hp_month[m].values
        Um = U_month[m].values
        Vm = V_month[m].values
        Pp = P_month[m].values

        # shading
        pcm = ax.pcolormesh(lon2d, lat2d, Hp, cmap="RdBu_r", norm=norm, shading="auto", transform=proj)
        pcm_ref = pcm

        # contours
        stepc = 5
        lev = np.arange(-max_abs, max_abs + stepc, stepc)
        pos = lev[lev > 0]
        neg = lev[lev < 0]

        if len(pos) > 0:
            ax.contour(lon2d, lat2d, Hp, levels=pos, colors="k", linewidths=0.7, transform=proj)
        if len(neg) > 0:
            ax.contour(lon2d, lat2d, Hp, levels=neg, colors="k", linewidths=0.7, linestyles="--", transform=proj)

        # vectors
        yy = np.arange(0, Um.shape[0], vector_step)
        xx = np.arange(0, Um.shape[1], vector_step)
        ax.quiver(lon2d[np.ix_(yy, xx)], lat2d[np.ix_(yy, xx)],
                  Um[np.ix_(yy, xx)], Vm[np.ix_(yy, xx)],
                  scale=700, width=0.002, color="black", transform=proj)

        # stippling (p-values)
        sig = np.isfinite(Pp) & (Pp < p_thresh)
        y, x = np.where(sig)
        y = y[::stipple_thin]; x = x[::stipple_thin]
        if len(y) > 0:
            ax.scatter(lon2d[y, x], lat2d[y, x], s=2, c="k", alpha=0.25, transform=proj)

        # VOM box + CDMX star
        rect = mpatches.Rectangle((VOM_BOX[0], VOM_BOX[1]), VOM_BOX[2], VOM_BOX[3],
                                  fill=False, edgecolor="k", linewidth=1, transform=proj)
        ax.add_patch(rect)
        ax.plot(LON_CDMX, LAT_CDMX, marker="*", color="gold", markersize=9,
                markeredgecolor="k", transform=proj)

        # Extract N_used for this month
        n_used = int(nctrl.loc[nctrl["month"] == m, "N_used"].values[0])
        
        # Title with month abbreviation + N_used + frac_area (for section 9)
        if frac_area_dict is not None:
            frac = frac_area_dict[m]
            ax.set_title(f"{calendar.month_abbr[m]} (N={n_used}, frac={frac*100:.1f}%)", 
                         fontsize=12, weight="bold")
        else:
            ax.set_title(f"{calendar.month_abbr[m]} (N={n_used})", 
                         fontsize=12, weight="bold")

        gl = ax.gridlines(draw_labels=True, linewidth=0.2, color="gray", alpha=0.5, linestyle="--")
        gl.top_labels = False
        gl.right_labels = False
        if (i % 4) != 0:
            gl.left_labels = False
        if i < 8:
            gl.bottom_labels = False

    # colorbar
    cbar_ax = fig.add_axes([0.92, 0.20, 0.015, 0.60])
    cb = fig.colorbar(pcm_ref, cax=cbar_ax)
    cb.set_label("Z500′ anomaly (m)")

    # titles
    fig.suptitle(f"{label}: {title_main} (2012–2024)", fontsize=18, weight="bold", y=0.96)
    fig.text(0.5, 0.91, subtitle, ha="center", fontsize=14, style="italic")

    legend_elements = [
        mpatches.Rectangle((0, 0), 1, 1, fill=False, edgecolor="k", linewidth=1.5, label="Valley of Mexico"),
        plt.Line2D([0], [0], marker="*", color="w", markerfacecolor="gold",
                   markeredgecolor="k", markersize=12, label="Mexico City (CDMX)")
    ]
    fig.legend(handles=legend_elements, loc="upper left", fontsize=11,
               frameon=False, bbox_to_anchor=(0.01, 0.97))

    plt.tight_layout(rect=[0, 0, 0.9, 0.93])
    plt.savefig(outpath, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"saved: {outpath}")

### 8.4 Save figures (p90, p10, p90-p10) for PM2.5 and PM10

In [92]:
# --- save figures (no stippling yet; P_month placeholders)
Pnan_pm25_p90 = _nan_pvals_like(Hp90_pm25)
Pnan_pm25_p10 = _nan_pvals_like(Hp10_pm25)
Pnan_pm25_dif = _nan_pvals_like(Hdif_pm25)

Pnan_pm10_p90 = _nan_pvals_like(Hp90_pm10)
Pnan_pm10_p10 = _nan_pvals_like(Hp10_pm10)
Pnan_pm10_dif = _nan_pvals_like(Hdif_pm10)

# PM2.5
plot_monthly_multipanel(
    label="PM2.5",
    Hp_month=Hp90_pm25, U_month=Up90_pm25, V_month=Vp90_pm25, P_month=Pnan_pm25_p90,
    max_abs=max90_pm25,
    outpath=OUT_FIG / "stage3_Z500_monthly_p90_PM25.png",
    title_main="Monthly Z500′ composites during p90 (high) days",
    subtitle="Shading/contours: Z500′ | Vectors: U′/V′ at 500 hPa | N balanced by month",
    nctrl=nctrl_pm25,
    vector_step=6
)

plot_monthly_multipanel(
    label="PM2.5",
    Hp_month=Hp10_pm25, U_month=Up10_pm25, V_month=Vp10_pm25, P_month=Pnan_pm25_p10,
    max_abs=max10_pm25,
    outpath=OUT_FIG / "stage3_Z500_monthly_p10_PM25.png",
    title_main="Monthly Z500′ composites during p10 (low) days",
    subtitle="Shading/contours: Z500′ | Vectors: U′/V′ at 500 hPa | N balanced by month",
    nctrl=nctrl_pm25,
    vector_step=6
)

plot_monthly_multipanel(
    label="PM2.5",
    Hp_month=Hdif_pm25, U_month=Udif_pm25, V_month=Vdif_pm25, P_month=Pnan_pm25_dif,
    max_abs=maxdf_pm25,
    outpath=OUT_FIG / "stage3_Z500_monthly_p90_minus_p10_PM25.png",
    title_main="Monthly Z500′ difference (p90 - p10)",
    subtitle="Shading/contours: ΔZ500′ | Vectors: Δ(U′,V′) at 500 hPa | N balanced by month",
    nctrl=nctrl_pm25,
    vector_step=6
)

# PM10
plot_monthly_multipanel(
    label="PM10",
    Hp_month=Hp90_pm10, U_month=Up90_pm10, V_month=Vp90_pm10, P_month=Pnan_pm10_p90,
    max_abs=max90_pm10,
    outpath=OUT_FIG / "stage3_Z500_monthly_p90_PM10.png",
    title_main="Monthly Z500′ composites during p90 (high) days",
    subtitle="Shading/contours: Z500′ | Vectors: U′/V′ at 500 hPa | N balanced by month",
    nctrl=nctrl_pm10,
    vector_step=6
)

plot_monthly_multipanel(
    label="PM10",
    Hp_month=Hp10_pm10, U_month=Up10_pm10, V_month=Vp10_pm10, P_month=Pnan_pm10_p10,
    max_abs=max10_pm10,
    outpath=OUT_FIG / "stage3_Z500_monthly_p10_PM10.png",
    title_main="Monthly Z500′ composites during p10 (low) days",
    subtitle="Shading/contours: Z500′ | Vectors: U′/V′ at 500 hPa | N balanced by month",
    nctrl=nctrl_pm10,
    vector_step=6
)

plot_monthly_multipanel(
    label="PM10",
    Hp_month=Hdif_pm10, U_month=Udif_pm10, V_month=Vdif_pm10, P_month=Pnan_pm10_dif,
    max_abs=maxdf_pm10,
    outpath=OUT_FIG / "stage3_Z500_monthly_p90_minus_p10_PM10.png",
    title_main="Monthly Z500′ difference (p90 - p10)",
    subtitle="Shading/contours: ΔZ500′ | Vectors: Δ(U′,V′) at 500 hPa | N balanced by month",
    nctrl=nctrl_pm10,
    vector_step=6
)

saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 3\outputs\figures\stage3_Z500_monthly_p90_PM25.png
saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 3\outputs\figures\stage3_Z500_monthly_p10_PM25.png
saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 3\outputs\figures\stage3_Z500_monthly_p90_minus_p10_PM25.png
saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 3\outputs\figures\stage3_Z500_monthly_p90_PM10.png
saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 3\outputs\figures\stage3_Z500_monthly_p10_PM10.png
saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 3\outputs\figures\stage3_Z500_monthly_p90_minus_p10_PM10.png


## 9. Significance testing on monthly difference maps (p90 - p10)

- Quantifying where the monthly differences between high (p90) and low (p10) composites are statistically robust.  
- Applying a gridpoint-wise two-sample t-test (Welch) comparing daily Z500′ during p90 vs p10 days, using balanced sample sizes (`N_used`) per month.  
- The resulting p-value masks are saved and used for stippling on the (p90 - p10) maps.

### 9.1 Helper: gridpoint Welch t-test p-values (p90 vs p10, balanced)

In [93]:
# --- Gridpoint significance test for each month, balanced N_used

def ttest_pvals_grid(da: xr.DataArray,
                     dates_a: pd.DatetimeIndex,
                     dates_b: pd.DatetimeIndex,
                     equal_var: bool = False) -> xr.DataArray:
    """
    Gridpoint two-sample t-test (Welch by default) between group A and B.
    Returns p-values with dims (y, x). da must have dims (time, y, x).
    """
    if (len(dates_a) == 0) or (len(dates_b) == 0):
        out = da.isel(time=0).drop_vars([v for v in da.isel(time=0).coords if v not in ["lat", "lon", "y", "x"]], errors="ignore") * np.nan
        return out

    A = da.sel(time=dates_a).values  # (na, y, x)
    B = da.sel(time=dates_b).values  # (nb, y, x)

    # scipy ttest over axis=0 (time axis)
    stat, p = ttest_ind(A, B, axis=0, equal_var=equal_var, nan_policy="omit")

    # wrap back into DataArray with lat/lon coords
    p_da = xr.DataArray(
        p,
        dims=("y", "x"),
        coords={"y": da["y"], "x": da["x"], "lat": da["lat"], "lon": da["lon"]},
        name="p_value"
    )
    return p_da


def monthly_pvals_balanced(pol: str, nctrl: pd.DataFrame) -> dict:
    """
    For each month, compute p-values for Z500′ difference between p90 and p10 days,
    using balanced date selection (same as Section 8).
    """
    P_month = {}
    for m in range(1, 13):
        n_used = _n_used_for_month(nctrl, m)
        d_hi = balanced_extreme_dates(df_evt, pol, m, "p90", n_used)
        d_lo = balanced_extreme_dates(df_evt, pol, m, "p10", n_used)

        pvals = ttest_pvals_grid(H_prime, d_hi, d_lo, equal_var=False)
        P_month[m] = pvals

        # diagnostic
        frac_sig = float((pvals < 0.05).mean(skipna=True).values)
        print(f"{pol} | month={m:02d} | N_used={n_used} | frac(p<0.05)={frac_sig:.3f}")

    return P_month

### 9.2 Compute monthly p-value masks (PM10 and PM2.5) + save diagnostics

In [94]:
# --- Compute p-values for p90 vs p10 (balanced)

Pvals_pm25 = monthly_pvals_balanced("PM2.5", nctrl_pm25)
Pvals_pm10 = monthly_pvals_balanced("PM10",  nctrl_pm10)

# Build a summary table: fraction significant per month
def frac_sig_table(P_month: dict, pol_label: str) -> pd.DataFrame:
    rows = []
    for m in range(1, 13):
        p = P_month[m]
        frac = float((p < 0.05).mean(skipna=True).values)
        rows.append([m, calendar.month_name[m], pol_label, frac])
    return pd.DataFrame(rows, columns=["month", "month_name", "pollutant", "frac_area_p_lt_0p05"])

sig_pm25 = frac_sig_table(Pvals_pm25, "PM2.5")
sig_pm10 = frac_sig_table(Pvals_pm10, "PM10")
sig_all = pd.concat([sig_pm25, sig_pm10], ignore_index=True)

out = OUT_TAB / "stage3_frac_significant_area_by_month.csv"
sig_all.to_csv(out, index=False)
print(f"saved: {out}")

sig_all

PM2.5 | month=01 | N_used=38 | frac(p<0.05)=0.388
PM2.5 | month=02 | N_used=36 | frac(p<0.05)=0.491
PM2.5 | month=03 | N_used=41 | frac(p<0.05)=0.499
PM2.5 | month=04 | N_used=39 | frac(p<0.05)=0.350
PM2.5 | month=05 | N_used=38 | frac(p<0.05)=0.354
PM2.5 | month=06 | N_used=36 | frac(p<0.05)=0.392
PM2.5 | month=07 | N_used=40 | frac(p<0.05)=0.305
PM2.5 | month=08 | N_used=40 | frac(p<0.05)=0.201
PM2.5 | month=09 | N_used=37 | frac(p<0.05)=0.331
PM2.5 | month=10 | N_used=38 | frac(p<0.05)=0.323
PM2.5 | month=11 | N_used=38 | frac(p<0.05)=0.441
PM2.5 | month=12 | N_used=40 | frac(p<0.05)=0.422
PM10 | month=01 | N_used=38 | frac(p<0.05)=0.419
PM10 | month=02 | N_used=36 | frac(p<0.05)=0.638
PM10 | month=03 | N_used=41 | frac(p<0.05)=0.379
PM10 | month=04 | N_used=39 | frac(p<0.05)=0.290
PM10 | month=05 | N_used=38 | frac(p<0.05)=0.338
PM10 | month=06 | N_used=36 | frac(p<0.05)=0.433
PM10 | month=07 | N_used=40 | frac(p<0.05)=0.018
PM10 | month=08 | N_used=41 | frac(p<0.05)=0.181
PM10 | m

,month,month_name,pollutant,frac_area_p_lt_0p05
0,1,January,PM2.5,0.39
1,2,February,PM2.5,0.49
2,3,March,PM2.5,0.50
3,4,April,PM2.5,0.35
4,5,May,PM2.5,0.35
5,6,June,PM2.5,0.39
6,7,July,PM2.5,0.30
7,8,August,PM2.5,0.20
8,9,September,PM2.5,0.33
9,10,October,PM2.5,0.32


### 9.3 Re-plot (p90 - p10) maps with stippling (final products)

In [108]:
# Build frac_area dicts from the significance tables (keyed by month)
frac_area_pm25 = dict(zip(sig_pm25["month"], sig_pm25["frac_area_p_lt_0p05"]))
frac_area_pm10 = dict(zip(sig_pm10["month"], sig_pm10["frac_area_p_lt_0p05"]))

In [109]:
# --- Final p90 - p10 plots

plot_monthly_multipanel(
    label="PM2.5",
    Hp_month=Hdif_pm25, U_month=Udif_pm25, V_month=Vdif_pm25, P_month=Pvals_pm25,
    max_abs=maxdf_pm25,
    outpath=OUT_FIG / "stage3_Z500_monthly_p90_minus_p10_PM25_STIPPLE.png",
    title_main="Monthly Z500′ difference (p90 - p10) with significance",
    subtitle="Shading/contours: ΔZ500′ | Vectors: Δ(U′,V′) at 500 hPa | Stippling: p<0.05 (Welch t-test) | N balanced by month",
    nctrl=nctrl_pm25,
    frac_area_dict=frac_area_pm25,
    vector_step=6
)

plot_monthly_multipanel(
    label="PM10",
    Hp_month=Hdif_pm10, U_month=Udif_pm10, V_month=Vdif_pm10, P_month=Pvals_pm10,
    max_abs=maxdf_pm10,
    outpath=OUT_FIG / "stage3_Z500_monthly_p90_minus_p10_PM10_STIPPLE.png",
    title_main="Monthly Z500′ difference (p90 - p10) with significance",
    subtitle="Shading/contours: ΔZ500′ | Vectors: Δ(U′,V′) at 500 hPa | Stippling: p<0.05 (Welch t-test) | N balanced by month",
    nctrl=nctrl_pm10,
    frac_area_dict=frac_area_pm10,
    vector_step=6
)

saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 3\outputs\figures\stage3_Z500_monthly_p90_minus_p10_PM25_STIPPLE.png
saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 3\outputs\figures\stage3_Z500_monthly_p90_minus_p10_PM10_STIPPLE.png


### 9.4 Appendix: Significance vs baseline (0 anomaly)

As an additional robustness check, let's test whether Z500′ composites during p90 and p10 days differ from the monthly baseline (0 anomaly) using a one-sample t-test at each grid point.  
In the end, a report of the fraction of significant area by month is computed, providing stippled maps for the appendix.

#### 9.4.1 One-sample t-test (p90 vs 0, p10 vs 0) + % significant area table

In [98]:
# --- One-sample t-test against 0 (baseline), per month, balanced N_used

def ttest1_pvals_grid(da: xr.DataArray,
                      dates: pd.DatetimeIndex,
                      popmean: float = 0.0) -> xr.DataArray:
    """
    Gridpoint one-sample t-test vs popmean (0 anomaly).
    Returns p-values with dims (y, x).
    """
    if len(dates) == 0:
        return xr.DataArray(
            np.full((da.sizes["y"], da.sizes["x"]), np.nan),
            dims=("y", "x"),
            coords={"y": da["y"], "x": da["x"], "lat": da["lat"], "lon": da["lon"]},
            name="p_value"
        )

    A = da.sel(time=dates).values  # (n, y, x)
    stat, p = ttest_1samp(A, popmean=popmean, axis=0, nan_policy="omit")

    return xr.DataArray(
        p,
        dims=("y", "x"),
        coords={"y": da["y"], "x": da["x"], "lat": da["lat"], "lon": da["lon"]},
        name="p_value"
    )


def monthly_pvals_vs0_balanced(pol: str, nctrl: pd.DataFrame, kind: str) -> dict:
    """
    For each month, compute p-values for H′ during p90 or p10 days vs 0.
    kind = "p90" or "p10"
    """
    P_month = {}
    for m in range(1, 13):
        n_used = _n_used_for_month(nctrl, m)
        d = balanced_extreme_dates(df_evt, pol, m, kind, n_used)
        pvals = ttest1_pvals_grid(H_prime, d, popmean=0.0)
        P_month[m] = pvals

        frac_sig = float((pvals < 0.05).mean(skipna=True).values)
        print(f"{pol} | {kind} vs 0 | month={m:02d} | N_used={n_used} | frac(p<0.05)={frac_sig:.3f}")
    return P_month

In [99]:
# Compute p-values vs 0 for p90 and p10 (PM2.5 and PM10)
P90vs0_pm25 = monthly_pvals_vs0_balanced("PM2.5", nctrl_pm25, "p90")
P10vs0_pm25 = monthly_pvals_vs0_balanced("PM2.5", nctrl_pm25, "p10")

P90vs0_pm10 = monthly_pvals_vs0_balanced("PM10",  nctrl_pm10, "p90")
P10vs0_pm10 = monthly_pvals_vs0_balanced("PM10",  nctrl_pm10, "p10")

PM2.5 | p90 vs 0 | month=01 | N_used=38 | frac(p<0.05)=0.043
PM2.5 | p90 vs 0 | month=02 | N_used=36 | frac(p<0.05)=0.472
PM2.5 | p90 vs 0 | month=03 | N_used=41 | frac(p<0.05)=0.000
PM2.5 | p90 vs 0 | month=04 | N_used=39 | frac(p<0.05)=0.140
PM2.5 | p90 vs 0 | month=05 | N_used=38 | frac(p<0.05)=0.356
PM2.5 | p90 vs 0 | month=06 | N_used=36 | frac(p<0.05)=0.257
PM2.5 | p90 vs 0 | month=07 | N_used=40 | frac(p<0.05)=0.279
PM2.5 | p90 vs 0 | month=08 | N_used=40 | frac(p<0.05)=0.191
PM2.5 | p90 vs 0 | month=09 | N_used=37 | frac(p<0.05)=0.228
PM2.5 | p90 vs 0 | month=10 | N_used=38 | frac(p<0.05)=0.381
PM2.5 | p90 vs 0 | month=11 | N_used=38 | frac(p<0.05)=0.154
PM2.5 | p90 vs 0 | month=12 | N_used=40 | frac(p<0.05)=0.347
PM2.5 | p10 vs 0 | month=01 | N_used=38 | frac(p<0.05)=0.404
PM2.5 | p10 vs 0 | month=02 | N_used=36 | frac(p<0.05)=0.346
PM2.5 | p10 vs 0 | month=03 | N_used=41 | frac(p<0.05)=0.617
PM2.5 | p10 vs 0 | month=04 | N_used=39 | frac(p<0.05)=0.254
PM2.5 | p10 vs 0 | month

In [100]:
def frac_sig_table_vs0(P_month: dict, pol_label: str, kind: str) -> pd.DataFrame:
    rows = []
    for m in range(1, 13):
        p = P_month[m]
        frac = float((p < 0.05).mean(skipna=True).values)
        rows.append([m, calendar.month_name[m], pol_label, kind, frac])
    return pd.DataFrame(rows, columns=["month", "month_name", "pollutant", "test", "frac_area_p_lt_0p05"])

sig_p90_0_pm25 = frac_sig_table_vs0(P90vs0_pm25, "PM2.5", "p90_vs_0")
sig_p10_0_pm25 = frac_sig_table_vs0(P10vs0_pm25, "PM2.5", "p10_vs_0")
sig_p90_0_pm10 = frac_sig_table_vs0(P90vs0_pm10, "PM10",  "p90_vs_0")
sig_p10_0_pm10 = frac_sig_table_vs0(P10vs0_pm10, "PM10",  "p10_vs_0")

sig_vs0_all = pd.concat([sig_p90_0_pm25, sig_p10_0_pm25, sig_p90_0_pm10, sig_p10_0_pm10], ignore_index=True)

out = OUT_TAB / "stage3_frac_significant_area_vs0_by_month.csv"
sig_vs0_all.to_csv(out, index=False)
print(f"saved: {out}")

sig_vs0_all

saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 3\outputs\tables\stage3_frac_significant_area_vs0_by_month.csv


,month,month_name,pollutant,test,frac_area_p_lt_0p05
0,1,January,PM2.5,p90_vs_0,0.04
1,2,February,PM2.5,p90_vs_0,0.47
2,3,March,PM2.5,p90_vs_0,0.00
3,4,April,PM2.5,p90_vs_0,0.14
4,5,May,PM2.5,p90_vs_0,0.36
5,6,June,PM2.5,p90_vs_0,0.26
6,7,July,PM2.5,p90_vs_0,0.28
7,8,August,PM2.5,p90_vs_0,0.19
8,9,September,PM2.5,p90_vs_0,0.23
9,10,October,PM2.5,p90_vs_0,0.38


#### 9.4.2 Appendix figures: p90 and p10 composites with stippling vs 0

Reusing the `plot_monthly_multipanel`(...) function and pass `P_month` as the p-values ​​vs. 0. (This adds stippling at p90/p10. These are figures for the appendix.)

In [110]:
# Build frac_area dicts from the significance tables (keyed by month)
frac_area_p90_0_pm25 = dict(zip(sig_p90_0_pm25["month"], sig_p90_0_pm25["frac_area_p_lt_0p05"]))
frac_area_p10_0_pm25 = dict(zip(sig_p10_0_pm25["month"], sig_p10_0_pm25["frac_area_p_lt_0p05"]))
frac_area_p90_0_pm10 = dict(zip(sig_p90_0_pm10["month"], sig_p90_0_pm10["frac_area_p_lt_0p05"]))
frac_area_p10_0_pm10 = dict(zip(sig_p10_0_pm10["month"], sig_p10_0_pm10["frac_area_p_lt_0p05"]))

In [112]:
# --- Appendix plots: p90 and p10 with stippling vs 0 (baseline)

# PM2.5 p90 vs 0
plot_monthly_multipanel(
    label="PM2.5",
    Hp_month=Hp90_pm25, U_month=Up90_pm25, V_month=Vp90_pm25, P_month=P90vs0_pm25,
    max_abs=max90_pm25,
    outpath=OUT_FIG / "stage3_Z500_monthly_p90_PM25_STIPPLE_vs0.png",
    title_main="Monthly Z500′ composites during p90 (high) days",
    subtitle="Shading/contours: Z500′ | Vectors: U′/V′ at 500 hPa | Stippling: p<0.05 (one-sample t-test vs 0) | N balanced by month",
    nctrl=nctrl_pm25,
    frac_area_dict=frac_area_p90_0_pm25,
    vector_step=6
)

# PM2.5 p10 vs 0
plot_monthly_multipanel(
    label="PM2.5",
    Hp_month=Hp10_pm25, U_month=Up10_pm25, V_month=Vp10_pm25, P_month=P10vs0_pm25,
    max_abs=max10_pm25,
    outpath=OUT_FIG / "stage3_Z500_monthly_p10_PM25_STIPPLE_vs0.png",
    title_main="Monthly Z500′ composites during p10 (low) days",
    subtitle="Shading/contours: Z500′ | Vectors: U′/V′ at 500 hPa | Stippling: p<0.05 (one-sample t-test vs 0) | N balanced by month",
    nctrl=nctrl_pm25,
    frac_area_dict=frac_area_p10_0_pm25,
    vector_step=6
)

# PM10 p90 vs 0
plot_monthly_multipanel(
    label="PM10",
    Hp_month=Hp90_pm10, U_month=Up90_pm10, V_month=Vp90_pm10, P_month=P90vs0_pm10,
    max_abs=max90_pm10,
    outpath=OUT_FIG / "stage3_Z500_monthly_p90_PM10_STIPPLE_vs0.png",
    title_main="Monthly Z500′ composites during p90 (high) days",
    subtitle="Shading/contours: Z500′ | Vectors: U′/V′ at 500 hPa | Stippling: p<0.05 (one-sample t-test vs 0) | N balanced by month",
    nctrl=nctrl_pm10,
    frac_area_dict=frac_area_p90_0_pm10,
    vector_step=6
)

# PM10 p10 vs 0
plot_monthly_multipanel(
    label="PM10",
    Hp_month=Hp10_pm10, U_month=Up10_pm10, V_month=Vp10_pm10, P_month=P10vs0_pm10,
    max_abs=max10_pm10,
    outpath=OUT_FIG / "stage3_Z500_monthly_p10_PM10_STIPPLE_vs0.png",
    title_main="Monthly Z500′ composites during p10 (low) days",
    subtitle="Shading/contours: Z500′ | Vectors: U′/V′ at 500 hPa | Stippling: p<0.05 (one-sample t-test vs 0) | N balanced by month",
    nctrl=nctrl_pm10,
    frac_area_dict=frac_area_p10_0_pm10,
    vector_step=6
)

saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 3\outputs\figures\stage3_Z500_monthly_p90_PM25_STIPPLE_vs0.png
saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 3\outputs\figures\stage3_Z500_monthly_p10_PM25_STIPPLE_vs0.png
saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 3\outputs\figures\stage3_Z500_monthly_p90_PM10_STIPPLE_vs0.png
saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 3\outputs\figures\stage3_Z500_monthly_p10_PM10_STIPPLE_vs0.png


### 9.5 Barplots of significance fractions (6-panel)

In [ ]:
# Extract series directly from sig_all and sig_vs0_all (computed in 9.2 and 9.4.1)
MONTH_LABELS = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

# From sig_all (p90-p10 difference)
s_diff_PM10 = sig_all[sig_all["pollutant"] == "PM10"]["frac_area_p_lt_0p05"].values * 100
s_diff_PM25 = sig_all[sig_all["pollutant"] == "PM2.5"]["frac_area_p_lt_0p05"].values * 100

# From sig_vs0_all (one-sample t-tests vs 0)
s_p90_PM10 = sig_vs0_all[(sig_vs0_all["pollutant"] == "PM10") & (sig_vs0_all["test"] == "p90_vs_0")]["frac_area_p_lt_0p05"].values * 100
s_p90_PM25 = sig_vs0_all[(sig_vs0_all["pollutant"] == "PM2.5") & (sig_vs0_all["test"] == "p90_vs_0")]["frac_area_p_lt_0p05"].values * 100
s_p10_PM10 = sig_vs0_all[(sig_vs0_all["pollutant"] == "PM10") & (sig_vs0_all["test"] == "p10_vs_0")]["frac_area_p_lt_0p05"].values * 100
s_p10_PM25 = sig_vs0_all[(sig_vs0_all["pollutant"] == "PM2.5") & (sig_vs0_all["test"] == "p10_vs_0")]["frac_area_p_lt_0p05"].values * 100

# Define custom colors for each test type
colors = {
    "p90": "#2E86AB",      # Deep blue
    "p10": "#E67E22",      # Orange
    "diff": "#229745"      # Green
}

# Plot 6-panel figure with enhanced styling
fig, axes = plt.subplots(3, 2, figsize=(16, 11), dpi=250, sharex=True, sharey=True)
fig.patch.set_facecolor("#FAFAF8")
x = np.arange(1, 13)

panel_data = [
    (axes[0, 0], s_p90_PM10,  "PM10 | p90 vs 0", colors["p90"]),
    (axes[0, 1], s_p90_PM25,  "PM2.5 | p90 vs 0", colors["p90"]),
    (axes[1, 0], s_p10_PM10,  "PM10 | p10 vs 0", colors["p10"]),
    (axes[1, 1], s_p10_PM25,  "PM2.5 | p10 vs 0", colors["p10"]),
    (axes[2, 0], s_diff_PM10, "PM10 | p90 - p10", colors["diff"]),
    (axes[2, 1], s_diff_PM25, "PM2.5 | p90 - p10", colors["diff"]),
]

for ax, y, title, color in panel_data:
    # Main bars with styled appearance
    bars = ax.bar(x, y, color=color, alpha=0.82, edgecolor="white", linewidth=1.5)
    
    # Subtle gradient effect with light overlay
    ax.bar(x, y, color=color, alpha=0.15, edgecolor="none", linewidth=0)
    
    # Add value labels on top of bars
    for xi, yi in zip(x, y):
        if yi > 0:
            ax.text(xi, yi + 1.5, f"{yi:.0f}", ha="center", va="bottom", 
                   fontsize=8.5, fontweight="600", color=color)
    
    # Panel styling
    ax.set_title(title, fontsize=12.5, fontweight="bold", pad=-8, color="#2C3E50")
    ax.set_xticks(x)
    ax.set_xticklabels(MONTH_LABELS, fontsize=10, fontweight="500")
    ax.set_ylim(0, 110)
    
    # Enhanced grid
    ax.grid(True, axis="y", alpha=0.25, linestyle="--", linewidth=0.7, color="#7F8C8D")
    ax.set_axisbelow(True)
    
    # Spine styling
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)
    ax.spines["left"].set_color("#BDC3C7")
    ax.spines["bottom"].set_color("#BDC3C7")
    ax.spines["left"].set_linewidth(1.2)
    ax.spines["bottom"].set_linewidth(1.2)
    
    # Y-axis label
    ax.set_ylabel("Significant area (%)", fontsize=10.5, fontweight="600", color="#34495E")
    ax.tick_params(axis="y", labelsize=9.5, colors="#34495E")
    ax.tick_params(axis="x", labelsize=10, colors="#34495E")
    
    # Subtle background
    ax.set_facecolor("#FCFCFA")

# Overall title with better styling
fig.suptitle("Monthly Fraction of Significant Grid Cells (p < 0.05)", 
            fontsize=18, fontweight="bold", y=0.98, color="#1A1A1A")

plt.subplots_adjust(hspace=0.12, wspace=0.25, top=0.93, bottom=0.08)

out = OUT_FIG / "stage3_sigfrac_barplots_6panel.png"
plt.savefig(out, dpi=250, bbox_inches="tight", facecolor=fig.get_facecolor(), edgecolor="none")
plt.close()
print(f"saved: {out}")

saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 3\outputs\figures\stage3_sigfrac_barplots_6panel.png


## 10. Summary tables + baseline figure

### 10.1 N-control and ranking tables

In [128]:
# Combined table (long format)
nctrl_long = pd.concat(
    [
        nctrl_pm10.assign(pollutant="PM10"),
        nctrl_pm25.assign(pollutant="PM2.5"),
    ],
    ignore_index=True
)[["pollutant", "month", "month_name", "n_available", "N_high_p90", "N_low_p10", "N_used"]]

out = OUT_TAB / "stage3_N_control_p90_vs_p10_ALL.csv"
nctrl_long.to_csv(out, index=False)
print(f"saved: {out}")

nctrl_long.head()

saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 3\outputs\tables\stage3_N_control_p90_vs_p10_ALL.csv


,pollutant,month,month_name,n_available,N_high_p90,N_low_p10,N_used
0,PM10,1,January,373,38,38,38
1,PM10,2,February,353,36,36,36
2,PM10,3,March,401,41,43,41
3,PM10,4,April,381,39,39,39
4,PM10,5,May,372,38,40,38


In [129]:
sig = pd.read_csv(OUT_TAB / "stage3_frac_significant_area_by_month.csv")

# month, month_name, pollutant, frac_area_p_lt_0p05
req = {"month", "month_name", "pollutant", "frac_area_p_lt_0p05"}
missing = req - set(sig.columns)
if missing:
    raise KeyError(f"Missing columns in significance table: {missing}. Found: {list(sig.columns)}")

sig["frac_percent"] = sig["frac_area_p_lt_0p05"] * 100.0

# Ranking within each pollutant
sig_rank = sig.sort_values(["pollutant", "frac_area_p_lt_0p05"], ascending=[True, False]).copy()
sig_rank["rank"] = sig_rank.groupby("pollutant").cumcount() + 1

out = OUT_TAB / "stage3_sigfrac_p90_minus_p10_ranking.csv"
sig_rank.to_csv(out, index=False)
print(f"saved: {out}")

sig_rank.head(15)

saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 3\outputs\tables\stage3_sigfrac_p90_minus_p10_ranking.csv


,month,month_name,pollutant,frac_area_p_lt_0p05,frac_percent,rank
13,2,February,PM10,0.64,63.83,1
17,6,June,PM10,0.43,43.35,2
12,1,January,PM10,0.42,41.87,3
14,3,March,PM10,0.38,37.87,4
16,5,May,PM10,0.34,33.76,5
23,12,December,PM10,0.32,32.24,6
22,11,November,PM10,0.31,30.98,7
15,4,April,PM10,0.29,28.95,8
19,8,August,PM10,0.18,18.10,9
20,9,September,PM10,0.11,10.78,10


In [130]:
top5 = sig_rank[sig_rank["rank"] <= 5].copy()
top5 = top5[["pollutant", "rank", "month", "month_name", "frac_percent"]]
top5["frac_percent"] = top5["frac_percent"].round(1)

out = OUT_TAB / "stage3_sigfrac_p90_minus_p10_top5.csv"
top5.to_csv(out, index=False)
print(f"saved: {out}")

top5

saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 3\outputs\tables\stage3_sigfrac_p90_minus_p10_top5.csv


,pollutant,rank,month,month_name,frac_percent
13,PM10,1,2,February,63.80
17,PM10,2,6,June,43.30
12,PM10,3,1,January,41.90
14,PM10,4,3,March,37.90
16,PM10,5,5,May,33.80
2,PM2.5,1,3,March,49.90
1,PM2.5,2,2,February,49.10
10,PM2.5,3,11,November,44.10
11,PM2.5,4,12,December,42.20
5,PM2.5,5,6,June,39.20


### 10.2 Baseline monthly Z500 (absolute) multipanel (12-panel) for context

In [133]:
def plot_monthly_multipanel_baseline(label: str,
                                     Hm_month: dict,
                                     Um_month: dict,
                                     Vm_month: dict,
                                     outpath: Path,
                                     title_main: str,
                                     subtitle: str,
                                     vector_step: int = 6):
    """
    Baseline (absolute) version of the Stage-2 style multipanel plot.
    Shading/contours: Z500 monthly mean (absolute, meters)
    Vectors: monthly mean winds (U, V) at 500 hPa
    No stippling.
    """
    proj = ccrs.PlateCarree()
    fig, axes = plt.subplots(3, 4, figsize=(16, 9), dpi=250, subplot_kw={"projection": proj})
    axes = axes.flatten()

    lon2d = H_prime["lon"].values
    lat2d = H_prime["lat"].values

    # color scale for absolute Z500 (robust)
    all_vals = np.concatenate([Hm_month[m].values.ravel() for m in range(1, 13)])
    all_vals = all_vals[np.isfinite(all_vals)]
    vmin = float(np.nanpercentile(all_vals, 1))
    vmax = float(np.nanpercentile(all_vals, 99))

    pcm_ref = None

    for i, m in enumerate(range(1, 13)):
        ax = axes[i]
        ax.set_extent([LON_MIN, LON_MAX, LAT_MIN, LAT_MAX], crs=proj)

        ax.coastlines(resolution="50m", linewidth=0.5)
        ax.add_feature(cfeature.BORDERS, linewidth=0.4)
        ax.add_feature(cfeature.STATES.with_scale("50m"), linewidth=0.3)

        Hm = Hm_month[m].values
        Um = Um_month[m].values
        Vm = Vm_month[m].values

        # shading (absolute Z500)
        pcm = ax.pcolormesh(lon2d, lat2d, Hm, cmap="viridis",
                            vmin=vmin, vmax=vmax, shading="auto", transform=proj)
        pcm_ref = pcm

        # contours (absolute)
        stepc = 30  # meters; tweak if you want denser/sparser
        lev = np.arange(np.floor(vmin/stepc)*stepc, np.ceil(vmax/stepc)*stepc + stepc, stepc)
        ax.contour(lon2d, lat2d, Hm, levels=lev, colors="k", linewidths=0.6, transform=proj)

        # vectors (mean winds)
        yy = np.arange(0, Um.shape[0], vector_step)
        xx = np.arange(0, Um.shape[1], vector_step)
        ax.quiver(lon2d[np.ix_(yy, xx)], lat2d[np.ix_(yy, xx)],
                  Um[np.ix_(yy, xx)], Vm[np.ix_(yy, xx)],
                  scale=700, width=0.002, color="black", transform=proj)

        # VOM box + CDMX star
        rect = mpatches.Rectangle((VOM_BOX[0], VOM_BOX[1]), VOM_BOX[2], VOM_BOX[3],
                                  fill=False, edgecolor="k", linewidth=1, transform=proj)
        ax.add_patch(rect)
        ax.plot(LON_CDMX, LAT_CDMX, marker="*", color="gold", markersize=9,
                markeredgecolor="k", transform=proj)

        ax.set_title(f"{calendar.month_abbr[m]}", fontsize=12, weight="bold")

        gl = ax.gridlines(draw_labels=True, linewidth=0.2, color="gray", alpha=0.5, linestyle="--")
        gl.top_labels = False
        gl.right_labels = False
        if (i % 4) != 0:
            gl.left_labels = False
        if i < 8:
            gl.bottom_labels = False

    # colorbar
    cbar_ax = fig.add_axes([0.92, 0.20, 0.015, 0.60])
    cb = fig.colorbar(pcm_ref, cax=cbar_ax)
    cb.set_label("Z500 monthly mean (m)")

    fig.suptitle(f"{label}: {title_main} (2012–2024)", fontsize=18, weight="bold", y=0.96)
    fig.text(0.5, 0.91, subtitle, ha="center", fontsize=14, style="italic")

    legend_elements = [
        mpatches.Rectangle((0, 0), 1, 1, fill=False, edgecolor="k", linewidth=1.5, label="Valley of Mexico"),
        plt.Line2D([0], [0], marker="*", color="w", markerfacecolor="gold",
                   markeredgecolor="k", markersize=12, label="Mexico City (CDMX)")
    ]
    fig.legend(handles=legend_elements, loc="upper left", fontsize=11,
               frameon=False, bbox_to_anchor=(0.01, 0.97))

    plt.tight_layout(rect=[0, 0, 0.9, 0.93])
    plt.savefig(outpath, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"saved: {outpath}")

In [134]:
# Build dicts from monthly climatologies (already computed in Section 5)
Hm_month = {m: H_clim_mon.sel(month=m) for m in range(1, 13)}
Um_month = {m: U_clim_mon.sel(month=m) for m in range(1, 13)}
Vm_month = {m: V_clim_mon.sel(month=m) for m in range(1, 13)}

out = OUT_FIG / "stage3_Z500_monthly_baseline_mean.png"
plot_monthly_multipanel_baseline(
    label="Baseline",
    Hm_month=Hm_month,
    Um_month=Um_month,
    Vm_month=Vm_month,
    outpath=out,
    title_main="Monthly Z500 climatology (baseline)",
    subtitle="Shading/contours: Z500 monthly mean | Vectors: monthly mean winds at 500 hPa"
)

saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 3\outputs\figures\stage3_Z500_monthly_baseline_mean.png


## Extra: WHO exceedance severity metrics (monthly)

In [139]:
# Panels:
# Row 1: % of days exceeding WHO (frequency) | PM10 (left), PM2.5 (right)
# Row 2: mean exceedance magnitude (µg/m³)   | PM10 (left), PM2.5 (right)
# Row 3: exceedance load (sum, µg/m³)        | PM10 (left), PM2.5 (right)

# Exceedance magnitude: mean(PM - WHO) over exceedance days
# Exceedance load: sum(PM - WHO) over exceedance days

WHO = {"PM10": 45.0, "PM2.5": 15.0}  # µg/m³ (24h)
months = np.arange(1, 13)
labels = [calendar.month_abbr[m] for m in months]

In [140]:
def who_exceedance_frequency(df: pd.DataFrame, pol: str, thr: float) -> pd.Series:
    """Monthly % of days exceeding WHO threshold (fixed), comparable across months."""
    s = df[pol].dropna()
    exc = s[s >= thr]
    exc_days = exc.groupby(exc.index.month).size().reindex(range(1, 13), fill_value=0)
    total_days = s.groupby(s.index.month).size().reindex(range(1, 13), fill_value=0)
    pct = (exc_days / total_days * 100).replace([np.inf, -np.inf], np.nan)
    return pct.astype(float)


def who_exceedance_metrics(df: pd.DataFrame, pol: str, thr: float) -> tuple[pd.Series, pd.Series]:
    """
    Monthly exceedance severity:
      - magnitude_mean: mean(PM - WHO) over exceedance days (µg/m³)
      - load_sum: sum(PM - WHO) over exceedance days (µg/m³)
    """
    s = df[pol].dropna()
    excess = (s - thr).where(s > thr)  # NaN on non-exceedance days
    mag = excess.groupby(excess.index.month).mean().reindex(range(1, 13))
    load = excess.groupby(excess.index.month).sum().reindex(range(1, 13)).fillna(0.0)
    return mag.astype(float), load.astype(float)

In [142]:
# Compute for both pollutants
pct_pm10 = who_exceedance_frequency(df_evt, "PM10", WHO["PM10"])
pct_pm25 = who_exceedance_frequency(df_evt, "PM2.5", WHO["PM2.5"])

mag_pm10, load_pm10 = who_exceedance_metrics(df_evt, "PM10", WHO["PM10"])
mag_pm25, load_pm25 = who_exceedance_metrics(df_evt, "PM2.5", WHO["PM2.5"])

# Save a combined CSV
who_tab = pd.DataFrame({
    "month": months,
    "month_name": [calendar.month_name[m] for m in months],

    "PM10_exceed_pct": pct_pm10.values,
    "PM10_excess_mag_mean": mag_pm10.values,
    "PM10_excess_load_sum": load_pm10.values,

    "PM25_exceed_pct": pct_pm25.values,
    "PM25_excess_mag_mean": mag_pm25.values,
    "PM25_excess_load_sum": load_pm25.values,
})

out_csv = OUT_TAB / "stage3_WHO_exceedance_frequency_magnitude_load_by_month.csv"
who_tab.to_csv(out_csv, index=False)
print(f"saved: {out_csv}")

who_tab

saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 3\outputs\tables\stage3_WHO_exceedance_frequency_magnitude_load_by_month.csv


,month,month_name,PM10_exceed_pct,PM10_excess_mag_mean,PM10_excess_load_sum,PM25_exceed_pct,PM25_excess_mag_mean,PM25_excess_load_sum
0,1,January,74.26,19.80,"5,444.50",87.67,14.87,"4,863.52"
1,2,February,72.80,17.19,"4,418.24",88.67,11.72,"3,666.80"
2,3,March,72.82,13.80,"3,987.90",91.52,10.11,"3,700.20"
3,4,April,69.03,15.01,"3,902.51",96.59,12.70,"4,634.77"
4,5,May,57.80,15.55,"3,297.54",97.04,14.28,"5,155.66"
5,6,June,22.22,7.16,551.32,73.06,7.89,"2,058.33"
6,7,July,14.82,5.23,308.32,78.14,6.18,"1,883.80"
7,8,August,7.32,6.71,181.17,63.38,5.05,"1,221.80"
8,9,September,8.11,6.83,204.83,64.05,6.47,"1,495.43"
9,10,October,22.55,9.22,783.33,62.33,7.73,"1,809.60"


In [154]:
# ---- 6-panel figure
colors = {
    "freq": "#66C2A5",     # Teal (from Set2)
    "mag": "#FC8D62",      # Orange (from Set2)
    "load": "#6A4C93"      # Dark purple
}

fig, axes = plt.subplots(3, 2, figsize=(15, 10), dpi=250)
fig.patch.set_facecolor("#FAFAF8")

def _bar(ax, x, y, title, ylabel, color, ylim=None):
    # Main bars with styled appearance
    bars = ax.bar(x, y, color=color, alpha=0.82, edgecolor="white", linewidth=1.5)
    
    # Subtle gradient effect with light overlay
    ax.bar(x, y, color=color, alpha=0.15, edgecolor="none", linewidth=0)
    
    # Add value labels on top of bars
    for xi, yi in zip(x, y):
        if yi > 0:
            ax.text(xi, yi + (max(y)*0.02), f"{yi:.0f}", ha="center", va="bottom", 
                   fontsize=7.5, fontweight="600", color=color)
    
    # Panel styling
    ax.set_title(title, fontsize=10, fontweight="bold", pad=5, color="#2C3E50")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=10, fontweight="500")
    
    # Enhanced grid
    ax.grid(True, axis="y", alpha=0.25, linestyle="--", linewidth=0.7, color="#7F8C8D")
    ax.set_axisbelow(True)
    
    # Spine styling
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)
    ax.spines["left"].set_color("#BDC3C7")
    ax.spines["bottom"].set_color("#BDC3C7")
    ax.spines["left"].set_linewidth(1.2)
    ax.spines["bottom"].set_linewidth(1.2)
    
    # Y-axis styling
    ax.set_ylabel(ylabel, fontsize=10.5, fontweight="600", color="#34495E", labelpad=10)
    ax.tick_params(axis="y", labelsize=9.5, colors="#34495E")
    ax.tick_params(axis="x", labelsize=10, colors="#34495E")
    
    # Subtle background
    ax.set_facecolor("#FCFCFA")
    
    if ylim is not None:
        ax.set_ylim(*ylim)

_bar(axes[0, 0], months, pct_pm10.values, f"PM10 | WHO exceedance frequency (thr={WHO['PM10']} µg/m³)", "% of days exceed", colors["freq"], ylim=(0, 120))
_bar(axes[0, 1], months, pct_pm25.values, f"PM2.5 | WHO exceedance frequency (thr={WHO['PM2.5']} µg/m³)", "% of days exceed", colors["freq"], ylim=(0, 120))
_bar(axes[0, 1], months, pct_pm25.values, f"PM2.5 | WHO exceedance frequency (thr={WHO['PM2.5']} µg/m³)", "% of days exceed", colors["freq"], ylim=(0, 110))

_bar(axes[1, 0], months, mag_pm10.values, "PM10 | Exceedance magnitude (mean PM-WHO on exceed days)", "µg/m³", colors["mag"], ylim=(0, max(mag_pm10)*1.20))
_bar(axes[1, 1], months, mag_pm25.values, "PM2.5 | Exceedance magnitude (mean PM-WHO on exceed days)", "µg/m³", colors["mag"], ylim=(0, max(mag_pm25)*1.20))
_bar(axes[1, 1], months, mag_pm25.values, "PM2.5 | Exceedance magnitude (mean PM-WHO on exceed days)", "µg/m³", colors["mag"])

_bar(axes[2, 0], months, load_pm10.values, "PM10 | Exceedance load (sum PM-WHO over exceed days)", "µg/m³", colors["load"], ylim=(0, max(load_pm10)*1.20))
_bar(axes[2, 1], months, load_pm25.values, "PM2.5 | Exceedance load (sum PM-WHO over exceed days)", "µg/m³", colors["load"], ylim=(0, max(load_pm25)*1.20))
_bar(axes[2, 1], months, load_pm25.values, "PM2.5 | Exceedance load (sum PM-WHO over exceed days)", "µg/m³", colors["load"])

fig.suptitle("WHO Exceedance Diagnostics by Month (2012-2024)", fontsize=16, fontweight="bold", y=0.98, color="#1A1A1A")
plt.subplots_adjust(hspace=0.4, wspace=0.25, top=0.93, bottom=0.08)

out_fig = OUT_FIG / "stage3_WHO_exceedance_6panel_frequency_magnitude_load.png"
plt.savefig(out_fig, dpi=250, bbox_inches="tight", facecolor=fig.get_facecolor(), edgecolor="none")
plt.close(fig)
print(f"saved: {out_fig}")

saved: c:\Users\DELL\OneDrive - TUNI.fi\Documents\Finlandia\Tampere Uni\Tesis\Schedule\Stage 3\outputs\figures\stage3_WHO_exceedance_6panel_frequency_magnitude_load.png
